# Public research notebook

This notebook is a cleaned public version of the original
research workflow.

## Execution model

- All filesystem paths are relative to the repository root.
- No external mounted filesystem is required.
- Stored cell outputs have been removed.
- Generated files are written below the local `results/`
  directory.
- The archival source notebook remains unchanged.


In [ ]:
# Portable repository configuration
#
# The notebook assumes that it is executed from the repository
# root or from a cloned copy of the repository.

from pathlib import Path

REPOSITORY_ROOT = Path.cwd().resolve()

# Move upward when the notebook is launched from a nested folder.
if REPOSITORY_ROOT.name in {
    "lorenz",
    "rossler",
    "duffing",
    "kuramoto",
    "stuart_landau",
    "coupled_map_lattice",
}:
    REPOSITORY_ROOT = REPOSITORY_ROOT.parents[1]

DATA_DIR = REPOSITORY_ROOT / "data"
RESULTS_DIR = REPOSITORY_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"

for directory in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    CHECKPOINTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPOSITORY_ROOT}")
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
# ============================================
# MOUNT GOOGLE DRIVE
# ============================================


# ============================================
# IMPORTY
# ============================================

import zipfile
import os
from glob import glob

# ============================================
# PATH TO THE ZIP DIRECTORY
# WARNING:
# The original folder name contained a trailing space
# "Testy ZIP "
# ============================================

ZIP_DIR = "data/raw"

# ============================================
# validation CO JEST W FOLDERZE
# ============================================

print("\nFILES IN ZIP DIRECTORY:\n")

!ls -lah "data/raw"

# ============================================
# SEARCH FOR ZIP ARCHIVES
# ============================================

zip_files = glob(ZIP_DIR + "*.zip")

print("\nFOUND ZIP FILES:\n")

for z in zip_files:
    print(z)

# ============================================
# FOLDER DO ROZPAKOWANIA
# ============================================

extract_base = "./recovered_delta_project"

os.makedirs(extract_base, exist_ok=True)

# ============================================
# ROZPAKOWANIE
# ============================================

for zfile in zip_files:

    name = os.path.splitext(os.path.basename(zfile))[0]

    out_dir = os.path.join(extract_base, name)

    os.makedirs(out_dir, exist_ok=True)

    try:

        with zipfile.ZipFile(zfile, "r") as zip_ref:
            zip_ref.extractall(out_dir)

        print(f"\nOK extracted: {name}")

    except Exception as e:

        print(f"\nERROR with {name}: {e}")

# ============================================
# PREVIEW OF RECOVERED FILES
# ============================================

print("\nRECOVERED FILES:\n")

!find ./recovered_delta_project | head -300

print("\nDONE")

In [ ]:
import zipfile
import os
from glob import glob

ZIP_DIR = "data/raw/"

zip_files = glob(ZIP_DIR + "*.zip")

print("FOUND:\n")

for z in zip_files:
    print(z)

extract_base = "./recovered_delta_project"

os.makedirs(extract_base, exist_ok=True)

for zfile in zip_files:

    name = os.path.splitext(os.path.basename(zfile))[0]

    out_dir = os.path.join(extract_base, name)

    os.makedirs(out_dir, exist_ok=True)

    try:

        with zipfile.ZipFile(zfile, "r") as zip_ref:
            zip_ref.extractall(out_dir)

        print(f"\nOK extracted: {name}")

    except Exception as e:

        print(f"\nERROR with {name}: {e}")

print("\nDONE")

In [ ]:
!find ./recovered_delta_project -type f | head -300

In [ ]:
!find ./recovered_delta_project -type f | grep ".csv"

In [ ]:
!find ./recovered_delta_project -type f | grep ".png"

In [ ]:
import shutil
import os

SRC = "./recovered_delta_project"
DST = "notebooks/Kuramoto/Stuart_Landau_recovered"

if os.path.exists(DST):
    shutil.rmtree(DST)

shutil.copytree(SRC, DST)

print("DONE")
print(DST)

In [ ]:

import os

BASE_DIR = "notebooks/Kuramoto/Stuart_Landau_new_tests"
os.makedirs(BASE_DIR, exist_ok=True)

print("Saving new tests to:")
print(BASE_DIR)

In [ ]:
# ============================================================
# STUART-LANDAU — MEMORY-LOCK EDGE v2
# core vs memory oscillators vs outsiders
# outputs saved directly to Google Drive
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# ============================================================
# OUTPUT
# ============================================================

BASE_DIR = "notebooks/Kuramoto/Stuart_Landau_new_tests"
output_dir = BASE_DIR + "/memory_lock_edge_v2"
os.makedirs(output_dir, exist_ok=True)

print("Saving to:", output_dir)

# ============================================================
# CONFIG
# ============================================================

N = 400
dt = 0.02
steps = 4000
discard = 2000

omega_std = 0.4
lambda_sl = 1.0

K_values = np.round(np.linspace(0.4, 1.3, 37), 3)
seeds = range(20)

memory_fraction = 0.10
outsider_fraction = 0.10

# ============================================================
# HELPERS
# ============================================================

def order_parameter(z):
    return np.abs(np.mean(np.exp(1j * np.angle(z))))

def freq_std(phases_t, dt):
    unwrapped = np.unwrap(phases_t, axis=0)
    inst_freq = np.diff(unwrapped, axis=0) / dt
    mean_freq = inst_freq.mean(axis=0)
    return np.std(mean_freq), mean_freq

# ============================================================
# MAIN
# ============================================================

rows = []

for seed in tqdm(seeds):

    rng = np.random.default_rng(seed)

    omega = rng.normal(0, omega_std, N)

    # memory oscillators = slow/central group
    sorted_idx = np.argsort(np.abs(omega))
    memory_n = int(N * memory_fraction)
    memory_idx = sorted_idx[:memory_n]

    # outsiders = far-frequency edge group
    outsider_n = int(N * outsider_fraction)
    outsider_idx = sorted_idx[-outsider_n:]

    core_idx = np.setdiff1d(
        np.arange(N),
        np.concatenate([memory_idx, outsider_idx])
    )

    for K in K_values:

        z = (
            rng.normal(0, 1, N)
            + 1j * rng.normal(0, 1, N)
        )

        phases_record = []

        for t in range(steps):

            mean_z = np.mean(z)

            dz = (
                (lambda_sl + 1j * omega - np.abs(z)**2) * z
                + K * (mean_z - z)
            )

            z += dt * dz

            if t >= discard:
                phases_record.append(np.angle(z))

        phases_record = np.array(phases_record)

        R = order_parameter(z)

        memory_freq_std, memory_freq = freq_std(phases_record[:, memory_idx], dt)
        core_freq_std, core_freq = freq_std(phases_record[:, core_idx], dt)
        outsider_freq_std, outsider_freq = freq_std(phases_record[:, outsider_idx], dt)

        memory_detuning = np.mean(np.abs(memory_freq - np.mean(core_freq)))
        core_detuning = np.mean(np.abs(core_freq - np.mean(core_freq)))
        outsider_detuning = np.mean(np.abs(outsider_freq - np.mean(core_freq)))

        memory_abs_omega = np.mean(np.abs(omega[memory_idx]))
        core_abs_omega = np.mean(np.abs(omega[core_idx]))
        outsider_abs_omega = np.mean(np.abs(omega[outsider_idx]))

        # score: memory remains close to core, outsiders remain separated
        memory_edge_score = (
            outsider_detuning / (memory_detuning + 1e-9)
        ) * R

        rows.append({
            "seed": seed,
            "K": K,
            "R": R,

            "memory_freq_std": memory_freq_std,
            "core_freq_std": core_freq_std,
            "outsider_freq_std": outsider_freq_std,

            "memory_detuning": memory_detuning,
            "core_detuning": core_detuning,
            "outsider_detuning": outsider_detuning,

            "memory_abs_omega": memory_abs_omega,
            "core_abs_omega": core_abs_omega,
            "outsider_abs_omega": outsider_abs_omega,

            "memory_edge_score": memory_edge_score
        })

# ============================================================
# DATAFRAMES
# ============================================================

df = pd.DataFrame(rows)

agg = (
    df.groupby("K")
    .agg(["mean", "std"])
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

summary = pd.DataFrame([{
    "peak_K_memory_edge": agg.loc[agg["memory_edge_score_mean"].idxmax(), "K"],
    "peak_memory_edge_score": agg["memory_edge_score_mean"].max(),
    "R_mean_at_peak": agg.loc[agg["memory_edge_score_mean"].idxmax(), "R_mean"],
    "memory_detuning_at_peak": agg.loc[agg["memory_edge_score_mean"].idxmax(), "memory_detuning_mean"],
    "core_detuning_at_peak": agg.loc[agg["memory_edge_score_mean"].idxmax(), "core_detuning_mean"],
    "outsider_detuning_at_peak": agg.loc[agg["memory_edge_score_mean"].idxmax(), "outsider_detuning_mean"]
}])

# ============================================================
# SAVE CSV
# ============================================================

df.to_csv(f"{output_dir}/memory_lock_edge_v2_all.csv", index=False)
agg.to_csv(f"{output_dir}/memory_lock_edge_v2_aggregate.csv", index=False)
summary.to_csv(f"{output_dir}/memory_lock_edge_v2_summary.csv", index=False)

# ============================================================
# PLOTS
# ============================================================

plt.figure(figsize=(10, 6))
plt.plot(agg["K"], agg["R_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("R mean")
plt.title("Global synchronization R")
plt.tight_layout()
plt.savefig(f"{output_dir}/fig_R_vs_K.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(agg["K"], agg["memory_detuning_mean"], marker="o", label="memory detuning")
plt.plot(agg["K"], agg["core_detuning_mean"], marker="o", label="core detuning")
plt.plot(agg["K"], agg["outsider_detuning_mean"], marker="o", label="outsider detuning")
plt.xlabel("K")
plt.ylabel("Detuning")
plt.title("Memory-core-outsider detuning")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig_detuning_vs_K.png", dpi=300)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(agg["K"], agg["memory_edge_score_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("Memory edge score")
plt.title("Memory-lock edge score")
plt.tight_layout()
plt.savefig(f"{output_dir}/fig_memory_edge_score.png", dpi=300)
plt.close()

# ============================================================
# PRINT
# ============================================================

print("\nSaved:")
print(output_dir)

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(40))

In [ ]:
import os
import pandas as pd

root = str(REPOSITORY_ROOT)

csv_files = []
png_files = []

for path, dirs, files in os.walk(root):
    for f in files:
        full = os.path.join(path, f)

        if f.endswith(".csv"):
            csv_files.append({
                "type": "csv",
                "name": f,
                "size_kb": round(os.path.getsize(full)/1024, 2),
                "path": full
            })

        if f.endswith(".png"):
            png_files.append({
                "type": "png",
                "name": f,
                "size_kb": round(os.path.getsize(full)/1024, 2),
                "path": full
            })

df_csv = pd.DataFrame(csv_files)
df_png = pd.DataFrame(png_files)

print("\nCSV FILES:")
display(df_csv.sort_values("name"))

print("\nPNG FILES:")
display(df_png.sort_values("name"))

In [ ]:
import os, shutil, hashlib
from pathlib import Path

sources = [
    "./recovered_delta_project",
    "notebooks/Kuramoto",
    "./delta_window_results",
]

master = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"
os.makedirs(master, exist_ok=True)

seen = set()
copied = []

def file_hash(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

for src in sources:
    for root, dirs, files in os.walk(src):
        for f in files:
            if not (f.endswith(".csv") or f.endswith(".png") or f.endswith(".docx") or f.endswith(".pdf")):
                continue

            full = os.path.join(root, f)
            try:
                h = file_hash(full)
            except:
                continue

            if h in seen:
                continue

            seen.add(h)

            rel = os.path.relpath(full, src)
            safe_rel = rel.replace(" ", "_")
            dst = os.path.join(master, Path(src).name, safe_rel)

            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(full, dst)

            copied.append(dst)

print("Copied unique files:", len(copied))
print("Saved to:", master)

for x in copied[:100]:
    print(x)

In [ ]:
import os
import pandas as pd
from pathlib import Path

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

rows = []

for path, dirs, files in os.walk(ROOT):
    for f in files:
        if f.endswith((".csv", ".png", ".docx", ".pdf")):
            full = os.path.join(path, f)
            rel = os.path.relpath(full, ROOT)

            parts = Path(rel).parts
            source = parts[0] if len(parts) > 0 else ""
            folder = parts[-2] if len(parts) > 1 else ""
            ext = Path(f).suffix.lower()

            rows.append({
                "source": source,
                "folder": folder,
                "filename": f,
                "extension": ext,
                "size_kb": round(os.path.getsize(full) / 1024, 2),
                "path": full
            })

index_df = pd.DataFrame(rows)

index_df = index_df.sort_values(["folder", "extension", "filename"]).reset_index(drop=True)

out_path = ROOT + "/MASTER_INDEX.csv"
index_df.to_csv(out_path, index=False)

print("MASTER INDEX saved:")
print(out_path)

print("\nTotal files:", len(index_df))
display(index_df)

In [ ]:
import pandas as pd
import os
from glob import glob

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

summary_files = glob(ROOT + "/**/*summary*.csv", recursive=True)

rows = []

for f in summary_files:
    if "sample_data" in f:
        continue

    try:
        df = pd.read_csv(f)
        folder = os.path.basename(os.path.dirname(f))

        row = {
            "test_folder": folder,
            "summary_file": os.path.basename(f),
            "path": f
        }

        if len(df) > 0:
            for col in df.columns:
                row[col] = df.iloc[0][col]

        rows.append(row)

    except Exception as e:
        rows.append({
            "test_folder": os.path.basename(os.path.dirname(f)),
            "summary_file": os.path.basename(f),
            "error": str(e),
            "path": f
        })

master_summary = pd.DataFrame(rows)

out = ROOT + "/MASTER_RESULTS_SUMMARY.csv"
master_summary.to_csv(out, index=False)

print("Saved:", out)
display(master_summary)

In [ ]:
import pandas as pd
import os

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

master_summary = pd.read_csv(ROOT + "/MASTER_RESULTS_SUMMARY.csv")

rows = []

for _, r in master_summary.iterrows():
    test = r["test_folder"]

    if test == "memory_lock_edge_v2":
        rows.append({
            "test": "Memory-lock edge v2",
            "main_peak_or_threshold": r.get("peak_K_memory_edge"),
            "main_metric": "memory/core locked while outsiders remain detuned",
            "key_value": f"memory detuning={r.get('memory_detuning_at_peak')}, outsider detuning={r.get('outsider_detuning_at_peak')}",
            "interpretation": "stable memory-core lock with outsider separation"
        })

    elif test == "stuart_landau_recovery_output":
        rows.append({
            "test": "Recovery after perturbation",
            "main_peak_or_threshold": r.get("peak_K_recovery_time"),
            "main_metric": "peak recovery time",
            "key_value": r.get("peak_recovery_time"),
            "interpretation": "critical recovery delay near reorganization window"
        })

    elif test == "stuart_landau_hysteresis_output":
        rows.append({
            "test": "Hysteresis",
            "main_peak_or_threshold": r.get("K_at_max_cluster_gap"),
            "main_metric": "max cluster hysteresis gap",
            "key_value": r.get("max_cluster_gap"),
            "interpretation": "path dependence / memory of previous collective state"
        })

    elif test == "stuart_landau_critical_slowing_output":
        rows.append({
            "test": "Critical slowing",
            "main_peak_or_threshold": r.get("peak_K_critical_slowing"),
            "main_metric": "critical slowing score",
            "key_value": r.get("peak_critical_slowing_score"),
            "interpretation": "early-warning-like slowing near transition"
        })

    elif test == "stuart_landau_basin_memory_output":
        rows.append({
            "test": "Basin memory",
            "main_peak_or_threshold": r.get("peak_K_basin_memory"),
            "main_metric": "basin memory score",
            "key_value": r.get("peak_basin_memory_score"),
            "interpretation": "initial-condition memory persists near transition"
        })

    elif test == "stuart_landau_outsider_rebellion_output":
        rows.append({
            "test": "Outsider rebellion",
            "main_peak_or_threshold": "see outsider_rebellion_summary.csv",
            "main_metric": "takeover / absorbed / fragmented / dual-cluster regimes",
            "key_value": "multi-parameter summary",
            "interpretation": "outsider faction can destabilize or capture global order"
        })

    elif "Delta_Window" in test:
        rows.append({
            "test": "Delta-window controls / paper package",
            "main_peak_or_threshold": r.get("delta_star"),
            "main_metric": "Δ* / controls / dwell-time / surrogate tests",
            "key_value": r.get("local_global_at_delta_star"),
            "interpretation": "dynamic reorganization window robust against controls"
        })

clean = pd.DataFrame(rows)

out = ROOT + "/MASTER_CLEAN_INTERPRETIVE_SUMMARY.csv"
clean.to_csv(out, index=False)

print("Saved:", out)
display(clean)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"
OUT = ROOT + "/cross_test_unification"
os.makedirs(OUT, exist_ok=True)

# ----------------------------
# Manual collection of peak values z MASTER_CLEAN_INTERPRETIVE_SUMMARY
# ----------------------------

peaks = pd.DataFrame([
    {
        "test": "Critical slowing",
        "K_peak": 0.45,
        "metric": "critical slowing score",
        "interpretation": "early warning / pre-transition slowing"
    },
    {
        "test": "Recovery delay",
        "K_peak": 0.575,
        "metric": "peak recovery time",
        "interpretation": "maximum recovery delay after perturbation"
    },
    {
        "test": "Basin memory",
        "K_peak": 0.65,
        "metric": "basin memory score",
        "interpretation": "initial-condition memory persists"
    },
    {
        "test": "Hysteresis",
        "K_peak": 0.675,
        "metric": "cluster hysteresis gap",
        "interpretation": "path dependence / state memory"
    },
    {
        "test": "Memory-lock edge",
        "K_peak": 0.70,
        "metric": "memory-core lock with outsider detuning",
        "interpretation": "memory/core lock while outsiders remain separated"
    }
])

peaks.to_csv(OUT + "/cross_test_peak_summary.csv", index=False)

display(peaks)

# ----------------------------
# Basic statistics of the reorganization window
# ----------------------------

window_summary = pd.DataFrame([{
    "K_min_peak": peaks["K_peak"].min(),
    "K_max_peak": peaks["K_peak"].max(),
    "K_mean_peak": peaks["K_peak"].mean(),
    "K_median_peak": peaks["K_peak"].median(),
    "K_std_peak": peaks["K_peak"].std(),
    "suggested_reorganization_window": f"{peaks['K_peak'].min():.3f} – {peaks['K_peak'].max():.3f}"
}])

window_summary.to_csv(OUT + "/cross_test_window_summary.csv", index=False)

display(window_summary)

# ----------------------------
# Figure 1: peak positions
# ----------------------------

plt.figure(figsize=(10, 5))

y = np.arange(len(peaks))

plt.scatter(peaks["K_peak"], y, s=120)
plt.yticks(y, peaks["test"])
plt.xlabel("K")
plt.title("Cross-test peak positions in Stuart–Landau pipeline")

plt.axvspan(peaks["K_peak"].min(), peaks["K_peak"].max(), alpha=0.15)
plt.axvline(peaks["K_peak"].mean(), linestyle="--", label=f"mean K={peaks['K_peak'].mean():.3f}")

plt.legend()
plt.tight_layout()
plt.savefig(OUT + "/fig_cross_test_peak_positions.png", dpi=300)
plt.show()

# ----------------------------
# Figure 2: phase-region sketch
# ----------------------------

regions = pd.DataFrame([
    {"region": "weak coupling / incoherent", "K_start": 0.40, "K_end": 0.45},
    {"region": "pre-transition slowing", "K_start": 0.45, "K_end": 0.575},
    {"region": "recovery-delay / compression", "K_start": 0.575, "K_end": 0.65},
    {"region": "memory / hysteresis window", "K_start": 0.65, "K_end": 0.70},
    {"region": "memory-lock / synchronization edge", "K_start": 0.70, "K_end": 0.80},
    {"region": "stable synchronized regime", "K_start": 0.80, "K_end": 1.30},
])

regions.to_csv(OUT + "/cross_test_phase_regions.csv", index=False)

plt.figure(figsize=(12, 3))

for i, row in regions.iterrows():
    plt.barh(
        y=0,
        width=row["K_end"] - row["K_start"],
        left=row["K_start"],
        height=0.5,
        label=row["region"]
    )

for _, row in peaks.iterrows():
    plt.axvline(row["K_peak"], linestyle="--", alpha=0.7)
    plt.text(row["K_peak"], 0.35, row["test"], rotation=90, va="bottom", ha="center", fontsize=8)

plt.yticks([])
plt.xlabel("K")
plt.title("Provisional Stuart–Landau reorganization phase diagram")
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2)
plt.tight_layout()
plt.savefig(OUT + "/fig_provisional_phase_diagram.png", dpi=300)
plt.show()

print("Saved to:")
print(OUT)

In [ ]:
# ============================================================
# STUART-LANDAU METASTABILITY LIFETIME — FULL NxN CHECKPOINT VERSION
# resume-safe: saves after every seed
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/stuart_landau_metastability_lifetime_FULL_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/metastability_lifetime_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))
K_VALUES = np.round(np.arange(0.4, 1.201, 0.05), 3)

omega_std = 0.4
dt = 0.03
steps = 5000
alpha = 1.0
cluster_threshold = 0.15

# ============================================================
# HELPERS
# ============================================================

def kuramoto_order(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def count_clusters(theta, threshold=0.15):
    theta_sorted = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(theta_sorted)
    circular_gap = (theta_sorted[0] + 2*np.pi) - theta_sorted[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def compute_run_lengths(arr):
    arr = np.asarray(arr)
    if len(arr) == 0:
        return []
    lengths = []
    current = arr[0]
    count = 1
    for x in arr[1:]:
        if x == current:
            count += 1
        else:
            lengths.append((current, count))
            current = x
            count = 1
    lengths.append((current, count))
    return lengths

# ============================================================
# LOAD EXISTING CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    done = set(zip(all_df["K"].round(3), all_df["seed"]))
    rows = all_df.to_dict("records")
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for K in K_VALUES:
    print(f"\nK={K}")

    for seed in tqdm(SEEDS):

        key = (round(float(K), 3), int(seed))

        if key in done:
            continue

        np.random.seed(seed)

        omega = np.random.normal(0, omega_std, N)

        x = np.random.normal(0, 0.5, N)
        y = np.random.normal(0, 0.5, N)

        R_series = []
        cluster_series = []

        for t in range(steps):

            z = x + 1j * y
            theta = np.angle(z)

            # FULL NxN coupling
            phase_diff = theta[:, None] - theta[None, :]
            coupling = K * np.mean(np.sin(phase_diff), axis=1)

            dz = (
                (alpha + 1j * omega - np.abs(z)**2) * z
                - 1j * coupling * z
            )

            z = z + dt * dz

            x = np.real(z)
            y = np.imag(z)

            theta = np.angle(z)

            R_series.append(kuramoto_order(theta))
            cluster_series.append(count_clusters(theta, cluster_threshold))

        R_series = np.array(R_series)
        cluster_series = np.array(cluster_series)

        run_lengths = compute_run_lengths(cluster_series)

        single_lifetimes = [l for state, l in run_lengths if state == 1]
        multi_lifetimes = [l for state, l in run_lengths if state > 1]

        dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
        dominant_fraction = np.mean(cluster_series == dominant_cluster)

        switches = np.sum(np.diff(cluster_series) != 0)
        switching_rate = switches / steps

        if np.any(R_series > 0.9):
            collapse_time = int(np.argmax(R_series > 0.9))
        else:
            collapse_time = np.nan

        memory_supported_lifetime = np.mean(
            pd.Series(cluster_series)
            .rolling(200)
            .std()
            .fillna(0)
            < 0.2
        )

        row = {
            "K": K,
            "seed": seed,
            "R_mean": np.mean(R_series),
            "R_std": np.std(R_series),
            "dominant_cluster": dominant_cluster,
            "dominant_fraction": dominant_fraction,
            "switching_rate": switching_rate,
            "collapse_time": collapse_time,
            "memory_supported_lifetime": memory_supported_lifetime,
            "cluster_mean": np.mean(cluster_series),
            "cluster_std": np.std(cluster_series),
            "single_cluster_lifetime_mean": np.mean(single_lifetimes) if len(single_lifetimes) else 0,
            "multi_cluster_lifetime_mean": np.mean(multi_lifetimes) if len(multi_lifetimes) else 0,
            "single_cluster_lifetime_max": np.max(single_lifetimes) if len(single_lifetimes) else 0,
            "multi_cluster_lifetime_max": np.max(multi_lifetimes) if len(multi_lifetimes) else 0,
            "n_switches": switches,
        }

        rows.append(row)
        done.add(key)

        # save checkpoint after every seed
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby("K")
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_supported_lifetime": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "single_cluster_lifetime_mean": ["mean", "std"],
        "multi_cluster_lifetime_mean": ["mean", "std"],
        "single_cluster_lifetime_max": ["mean", "std"],
        "multi_cluster_lifetime_max": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

peak_idx = agg["memory_supported_lifetime_mean"].idxmax()

summary = pd.DataFrame([{
    "peak_K_memory_supported_lifetime": agg.loc[peak_idx, "K"],
    "peak_memory_supported_lifetime": agg.loc[peak_idx, "memory_supported_lifetime_mean"],
    "switching_rate_at_peak": agg.loc[peak_idx, "switching_rate_mean"],
    "dominant_fraction_at_peak": agg.loc[peak_idx, "dominant_fraction_mean"],
    "cluster_mean_at_peak": agg.loc[peak_idx, "cluster_mean_mean"],
    "single_cluster_lifetime_mean_at_peak": agg.loc[peak_idx, "single_cluster_lifetime_mean_mean"],
    "multi_cluster_lifetime_mean_at_peak": agg.loc[peak_idx, "multi_cluster_lifetime_mean_mean"],
}])

# ============================================================
# SAVE FINAL
# ============================================================

all_df.to_csv(OUT + "/metastability_lifetime_all.csv", index=False)
agg.to_csv(OUT + "/metastability_lifetime_aggregate.csv", index=False)
summary.to_csv(OUT + "/metastability_lifetime_summary.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["memory_supported_lifetime_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("Memory-supported lifetime")
plt.title("Metastability: memory-supported lifetime vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_memory_supported_lifetime_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["switching_rate_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Metastability: cluster switching rate vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_switching_rate_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["dominant_fraction_mean"], marker="o")
plt.xlabel("K")
plt.ylabel("Dominant fraction")
plt.title("Metastability: dominant state fraction vs K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_dominant_fraction_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["single_cluster_lifetime_mean_mean"], marker="o", label="single cluster")
plt.plot(agg["K"], agg["multi_cluster_lifetime_mean_mean"], marker="o", label="multi cluster")
plt.xlabel("K")
plt.ylabel("Mean lifetime")
plt.title("Single vs multi-cluster lifetime")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_single_vs_multi_cluster_lifetime.png", dpi=300)
plt.show()

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE")
display(agg)

print("\nSaved to:")
print(OUT)

In [ ]:
!ls -lah "notebooks/Kuramoto/stuart_landau_metastability_lifetime_FULL_CHECKPOINT"

In [ ]:
# =========================
# ADD NEW TEST TO MASTER ARCHIVE
# =========================

import os
import shutil

SOURCE = "notebooks/Kuramoto/stuart_landau_metastability_lifetime_FULL_CHECKPOINT"
TARGET = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE/stuart_landau_metastability_lifetime"

os.makedirs(TARGET, exist_ok=True)

for f in os.listdir(SOURCE):
    src = os.path.join(SOURCE, f)
    dst = os.path.join(TARGET, f)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("DONE")
print(TARGET)

In [ ]:
# =========================
# REBUILD MASTER INDEX
# =========================

import os
import pandas as pd
from pathlib import Path

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

rows = []

for path, dirs, files in os.walk(ROOT):
    for f in files:
        if f.endswith((".csv", ".png", ".docx", ".pdf")):
            full = os.path.join(path, f)
            rel = os.path.relpath(full, ROOT)

            parts = Path(rel).parts
            source = parts[0] if len(parts) > 0 else ""
            folder = parts[-2] if len(parts) > 1 else ""
            ext = Path(f).suffix.lower()

            rows.append({
                "source": source,
                "folder": folder,
                "filename": f,
                "extension": ext,
                "size_kb": round(os.path.getsize(full) / 1024, 2),
                "path": full
            })

index_df = pd.DataFrame(rows)
index_df = index_df.sort_values(["folder", "extension", "filename"]).reset_index(drop=True)

out_path = ROOT + "/MASTER_INDEX.csv"
index_df.to_csv(out_path, index=False)

summary = (
    index_df
    .groupby(["folder", "extension"])
    .size()
    .reset_index(name="file_count")
    .sort_values(["folder", "extension"])
)

summary_path = ROOT + "/MASTER_INDEX_SUMMARY.csv"
summary.to_csv(summary_path, index=False)

print("MASTER_INDEX saved:", out_path)
print("MASTER_INDEX_SUMMARY saved:", summary_path)
print("Total files:", len(index_df))

display(summary)

In [ ]:
# ============================================================
# STUART-LANDAU ADAPTIVE COUPLING v1 — CHECKPOINT VERSION
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# ============================================================
# OUTPUT
# ============================================================

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/stuart_landau_adaptive_coupling_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/adaptive_coupling_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))
K_VALUES = np.round(np.arange(0.4, 1.201, 0.05), 3)

omega_std = 0.4

dt = 0.03
steps = 4000
discard = 1500

alpha = 1.0

# adaptive coupling parameters
eta = 0.004          # learning rate
decay = 0.001        # weakening / forgetting
K_min = 0.0
K_max = 2.5

cluster_threshold = 0.15

# to reduce memory: Kij is float32
dtype = np.float32

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return np.abs(np.mean(np.exp(1j * theta)))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def safe_mean(x):
    return float(np.mean(x)) if len(x) else np.nan

def compute_network_stats(Kij):
    upper = Kij[np.triu_indices_from(Kij, k=1)]
    return {
        "Kij_mean": float(np.mean(upper)),
        "Kij_std": float(np.std(upper)),
        "Kij_min": float(np.min(upper)),
        "Kij_max": float(np.max(upper)),
        "Kij_density_weak": float(np.mean(upper < 0.05)),
        "Kij_density_strong": float(np.mean(upper > 1.0)),
    }

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(all_df["K"].round(3), all_df["seed"]))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for K_base in K_VALUES:

    print(f"\nK_base={K_base}")

    for seed in tqdm(SEEDS):

        key = (round(float(K_base), 3), int(seed))

        if key in done:
            continue

        rng = np.random.default_rng(seed)

        omega = rng.normal(0, omega_std, N)

        z = (
            rng.normal(0, 0.5, N)
            + 1j * rng.normal(0, 0.5, N)
        )

        # initial adaptive couplings around K_base/N scaling
        Kij = np.full((N, N), K_base, dtype=dtype)
        np.fill_diagonal(Kij, 0.0)

        R_series = []
        cluster_series = []
        Kij_mean_series = []
        Kij_std_series = []

        for t in range(steps):

            theta = np.angle(z)

            phase_diff = theta[None, :] - theta[:, None]

            # coupling input: sum_j Kij sin(theta_j - theta_i) / N
            coupling = np.sum(Kij * np.sin(phase_diff), axis=1) / N

            dz = (
                (alpha + 1j * omega - np.abs(z)**2) * z
                + 1j * coupling * z
            )

            z = z + dt * dz

            # adaptive rule:
            # phase-compatible links strengthen, incompatible links weaken
            theta = np.angle(z)
            phase_similarity = np.cos(theta[None, :] - theta[:, None])

            dK = eta * phase_similarity - decay * Kij
            Kij = Kij + dK.astype(dtype)
            Kij = np.clip(Kij, K_min, K_max)
            np.fill_diagonal(Kij, 0.0)

            if t >= discard:
                R_series.append(order_parameter(theta))
                cluster_series.append(count_clusters(theta, cluster_threshold))

                if t % 50 == 0:
                    upper = Kij[np.triu_indices_from(Kij, k=1)]
                    Kij_mean_series.append(float(np.mean(upper)))
                    Kij_std_series.append(float(np.std(upper)))

        R_series = np.array(R_series)
        cluster_series = np.array(cluster_series)

        net_stats = compute_network_stats(Kij)

        switches = int(np.sum(np.diff(cluster_series) != 0))
        switching_rate = switches / len(cluster_series)

        dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
        dominant_fraction = float(np.mean(cluster_series == dominant_cluster))

        row = {
            "K": K_base,
            "seed": seed,

            "R_mean": float(np.mean(R_series)),
            "R_std": float(np.std(R_series)),

            "cluster_mean": float(np.mean(cluster_series)),
            "cluster_std": float(np.std(cluster_series)),
            "dominant_cluster": int(dominant_cluster),
            "dominant_fraction": dominant_fraction,
            "switching_rate": switching_rate,
            "n_switches": switches,

            "Kij_mean_final": net_stats["Kij_mean"],
            "Kij_std_final": net_stats["Kij_std"],
            "Kij_min_final": net_stats["Kij_min"],
            "Kij_max_final": net_stats["Kij_max"],
            "Kij_density_weak": net_stats["Kij_density_weak"],
            "Kij_density_strong": net_stats["Kij_density_strong"],

            "Kij_mean_time": safe_mean(Kij_mean_series),
            "Kij_std_time": safe_mean(Kij_std_series),
        }

        rows.append(row)
        done.add(key)

        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby("K")
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "Kij_mean_final": ["mean", "std"],
        "Kij_std_final": ["mean", "std"],
        "Kij_density_weak": ["mean", "std"],
        "Kij_density_strong": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

# adaptive organization score
agg["adaptive_reorganization_score"] = (
    agg["Kij_std_final_mean"]
    * agg["cluster_std_mean"]
    * (1 - agg["dominant_fraction_mean"])
)

peak_idx = agg["adaptive_reorganization_score"].idxmax()

summary = pd.DataFrame([{
    "peak_K_adaptive_reorganization": agg.loc[peak_idx, "K"],
    "peak_adaptive_reorganization_score": agg.loc[peak_idx, "adaptive_reorganization_score"],
    "R_mean_at_peak": agg.loc[peak_idx, "R_mean_mean"],
    "cluster_mean_at_peak": agg.loc[peak_idx, "cluster_mean_mean"],
    "dominant_fraction_at_peak": agg.loc[peak_idx, "dominant_fraction_mean"],
    "switching_rate_at_peak": agg.loc[peak_idx, "switching_rate_mean"],
    "Kij_mean_final_at_peak": agg.loc[peak_idx, "Kij_mean_final_mean"],
    "Kij_std_final_at_peak": agg.loc[peak_idx, "Kij_std_final_mean"],
    "Kij_density_strong_at_peak": agg.loc[peak_idx, "Kij_density_strong_mean"],
}])

# ============================================================
# SAVE FINAL
# ============================================================

all_df.to_csv(OUT + "/adaptive_coupling_all.csv", index=False)
agg.to_csv(OUT + "/adaptive_coupling_aggregate.csv", index=False)
summary.to_csv(OUT + "/adaptive_coupling_summary.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["R_mean_mean"], marker="o")
plt.xlabel("K base")
plt.ylabel("R mean")
plt.title("Adaptive coupling: global synchronization")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_adaptive_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["cluster_mean_mean"], marker="o")
plt.xlabel("K base")
plt.ylabel("Cluster mean")
plt.title("Adaptive coupling: cluster count")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_adaptive_cluster_mean_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["Kij_mean_final_mean"], marker="o", label="Kij mean")
plt.plot(agg["K"], agg["Kij_std_final_mean"], marker="o", label="Kij std")
plt.xlabel("K base")
plt.ylabel("Adaptive coupling statistics")
plt.title("Adaptive coupling network structure")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_adaptive_Kij_stats_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.plot(agg["K"], agg["adaptive_reorganization_score"], marker="o")
plt.xlabel("K base")
plt.ylabel("Adaptive reorganization score")
plt.title("Adaptive coupling: reorganization score")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_adaptive_reorganization_score.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE")
display(agg)

print("\nSaved to:")
print(OUT)

In [ ]:
import os
import shutil

SOURCE = "notebooks/Kuramoto/stuart_landau_adaptive_coupling_v1_CHECKPOINT"
TARGET = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE/stuart_landau_adaptive_coupling_v1"

os.makedirs(TARGET, exist_ok=True)

for f in os.listdir(SOURCE):
    src = os.path.join(SOURCE, f)
    dst = os.path.join(TARGET, f)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("DONE")
print(TARGET)

In [ ]:
# =========================
# REBUILD MASTER INDEX AFTER ADAPTIVE COUPLING
# =========================

import os
import pandas as pd
from pathlib import Path

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

rows = []

for path, dirs, files in os.walk(ROOT):
    for f in files:
        if f.endswith((".csv", ".png", ".docx", ".pdf")):
            full = os.path.join(path, f)
            rel = os.path.relpath(full, ROOT)

            parts = Path(rel).parts
            source = parts[0] if len(parts) > 0 else ""
            folder = parts[-2] if len(parts) > 1 else ""
            ext = Path(f).suffix.lower()

            rows.append({
                "source": source,
                "folder": folder,
                "filename": f,
                "extension": ext,
                "size_kb": round(os.path.getsize(full) / 1024, 2),
                "path": full
            })

index_df = pd.DataFrame(rows)
index_df = index_df.sort_values(["folder", "extension", "filename"]).reset_index(drop=True)

index_path = ROOT + "/MASTER_INDEX.csv"
index_df.to_csv(index_path, index=False)

summary = (
    index_df
    .groupby(["folder", "extension"])
    .size()
    .reset_index(name="file_count")
    .sort_values(["folder", "extension"])
)

summary_path = ROOT + "/MASTER_INDEX_SUMMARY.csv"
summary.to_csv(summary_path, index=False)

print("MASTER_INDEX saved:", index_path)
print("MASTER_INDEX_SUMMARY saved:", summary_path)
print("Total files:", len(index_df))

display(summary)

In [ ]:
# ============================================================
# STUART-LANDAU NETWORK TOPOLOGY ROBUSTNESS v1 — CHECKPOINT
# topologies: all-to-all, small-world, scale-free, modular, lattice
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from scipy import sparse

# ============================================================
# OUTPUT
# ============================================================

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/stuart_landau_network_topology_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/network_topology_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))
K_VALUES = np.round(np.arange(0.4, 1.201, 0.05), 3)

omega_std = 0.4

dt = 0.03
steps = 3000
discard = 1000

alpha = 1.0
cluster_threshold = 0.15

TOPOLOGIES = [
    "all_to_all",
    "small_world",
    "scale_free",
    "modular",
    "spatial_lattice"
]

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def normalize_adjacency(A):
    deg = np.array(A.sum(axis=1)).flatten()
    deg[deg == 0] = 1.0
    D_inv = sparse.diags(1.0 / deg)
    return D_inv @ A

def make_small_world(N, k=8, p=0.08, seed=0):
    rng = np.random.default_rng(seed)
    rows, cols = [], []

    for i in range(N):
        for d in range(1, k//2 + 1):
            j1 = (i + d) % N
            j2 = (i - d) % N
            rows.extend([i, i])
            cols.extend([j1, j2])

    edges = list(zip(rows, cols))
    new_edges = []

    for i, j in edges:
        if rng.random() < p:
            new_j = rng.integers(0, N)
            while new_j == i:
                new_j = rng.integers(0, N)
            new_edges.append((i, new_j))
        else:
            new_edges.append((i, j))

    r, c = zip(*new_edges)
    data = np.ones(len(r))
    A = sparse.csr_matrix((data, (r, c)), shape=(N, N))
    A = ((A + A.T) > 0).astype(float)
    A.setdiag(0)
    return normalize_adjacency(A.tocsr())

def make_scale_free(N, m=4, seed=0):
    rng = np.random.default_rng(seed)

    degrees = np.ones(m + 1)
    edges = []

    # initial clique
    for i in range(m + 1):
        for j in range(i + 1, m + 1):
            edges.append((i, j))
            edges.append((j, i))

    degrees = np.zeros(N)
    for i, j in edges:
        degrees[i] += 1

    for new_node in range(m + 1, N):
        probs = degrees[:new_node] + 1e-9
        probs = probs / probs.sum()

        targets = rng.choice(
            np.arange(new_node),
            size=m,
            replace=False,
            p=probs
        )

        for t in targets:
            edges.append((new_node, t))
            edges.append((t, new_node))
            degrees[new_node] += 1
            degrees[t] += 1

    r, c = zip(*edges)
    data = np.ones(len(r))
    A = sparse.csr_matrix((data, (r, c)), shape=(N, N))
    A.setdiag(0)
    return normalize_adjacency(A.tocsr())

def make_modular(N, modules=4, p_in=0.08, p_out=0.008, seed=0):
    rng = np.random.default_rng(seed)
    module_size = N // modules

    rows, cols = [], []

    labels = np.repeat(np.arange(modules), module_size)
    if len(labels) < N:
        labels = np.concatenate([labels, np.full(N-len(labels), modules-1)])

    for i in range(N):
        for j in range(i + 1, N):
            p = p_in if labels[i] == labels[j] else p_out
            if rng.random() < p:
                rows.extend([i, j])
                cols.extend([j, i])

    data = np.ones(len(rows))
    A = sparse.csr_matrix((data, (rows, cols)), shape=(N, N))
    A.setdiag(0)
    return normalize_adjacency(A.tocsr())

def make_spatial_lattice(N, k=4):
    rows, cols = [], []

    side = int(np.sqrt(N))
    assert side * side == N, "N must be perfect square for lattice"

    def idx(x, y):
        return (x % side) * side + (y % side)

    for x in range(side):
        for y in range(side):
            i = idx(x, y)

            neigh = [
                idx(x+1, y),
                idx(x-1, y),
                idx(x, y+1),
                idx(x, y-1)
            ]

            for j in neigh:
                rows.append(i)
                cols.append(j)

    data = np.ones(len(rows))
    A = sparse.csr_matrix((data, (rows, cols)), shape=(N, N))
    A.setdiag(0)
    return normalize_adjacency(A.tocsr())

def get_network(topology, N, seed):
    if topology == "all_to_all":
        return None

    if topology == "small_world":
        return make_small_world(N, k=8, p=0.08, seed=seed)

    if topology == "scale_free":
        return make_scale_free(N, m=4, seed=seed)

    if topology == "modular":
        return make_modular(N, modules=4, p_in=0.08, p_out=0.008, seed=seed)

    if topology == "spatial_lattice":
        return make_spatial_lattice(N, k=4)

    raise ValueError("Unknown topology")

def rolling_memory_score(cluster_series, window=200):
    s = pd.Series(cluster_series)
    return float((s.rolling(window).std().fillna(0) < 0.2).mean())

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(all_df["topology"], all_df["K"].round(3), all_df["seed"]))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for topology in TOPOLOGIES:

    print("\n" + "="*70)
    print("TOPOLOGY:", topology)
    print("="*70)

    for K in K_VALUES:

        print(f"\nK={K}")

        for seed in tqdm(SEEDS):

            key = (topology, round(float(K), 3), int(seed))

            if key in done:
                continue

            rng = np.random.default_rng(seed)

            omega = rng.normal(0, omega_std, N)

            z = (
                rng.normal(0, 0.5, N)
                + 1j * rng.normal(0, 0.5, N)
            )

            A = get_network(topology, N, seed)

            R_series = []
            cluster_series = []

            for t in range(steps):

                if topology == "all_to_all":
                    mean_z = np.mean(z)
                    coupling = K * (mean_z - z)

                else:
                    neighbor_mean = A @ z
                    coupling = K * (neighbor_mean - z)

                dz = (
                    (alpha + 1j * omega - np.abs(z)**2) * z
                    + coupling
                )

                z = z + dt * dz

                if t >= discard:
                    theta = np.angle(z)
                    R_series.append(order_parameter(theta))
                    cluster_series.append(count_clusters(theta, cluster_threshold))

            R_series = np.array(R_series)
            cluster_series = np.array(cluster_series)

            dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
            dominant_fraction = float(np.mean(cluster_series == dominant_cluster))

            switching_rate = float(np.sum(np.diff(cluster_series) != 0) / len(cluster_series))

            memory_score = rolling_memory_score(cluster_series, window=200)

            row = {
                "topology": topology,
                "K": K,
                "seed": seed,

                "R_mean": float(np.mean(R_series)),
                "R_std": float(np.std(R_series)),

                "cluster_mean": float(np.mean(cluster_series)),
                "cluster_std": float(np.std(cluster_series)),

                "dominant_cluster": int(dominant_cluster),
                "dominant_fraction": dominant_fraction,
                "switching_rate": switching_rate,
                "memory_score": memory_score,

                "reorganization_score": float(
                    np.std(cluster_series)
                    * (1 - dominant_fraction)
                    * (switching_rate + 1e-9)
                )
            }

            rows.append(row)
            done.add(key)

            pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby(["topology", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_score": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

# ============================================================
# SUMMARY
# ============================================================

summary_rows = []

for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology].copy()

    idx = sub["reorganization_score_mean"].idxmax()

    summary_rows.append({
        "topology": topology,
        "peak_K_reorganization": sub.loc[idx, "K"],
        "peak_reorganization_score": sub.loc[idx, "reorganization_score_mean"],
        "R_mean_at_peak": sub.loc[idx, "R_mean_mean"],
        "cluster_mean_at_peak": sub.loc[idx, "cluster_mean_mean"],
        "dominant_fraction_at_peak": sub.loc[idx, "dominant_fraction_mean"],
        "switching_rate_at_peak": sub.loc[idx, "switching_rate_mean"],
        "memory_score_at_peak": sub.loc[idx, "memory_score_mean"],
    })

summary = pd.DataFrame(summary_rows)

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/network_topology_all.csv", index=False)
agg.to_csv(OUT + "/network_topology_aggregate.csv", index=False)
summary.to_csv(OUT + "/network_topology_summary.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["R_mean_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("R mean")
plt.title("Network topology robustness: global synchronization")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_topology_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["cluster_mean_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("Cluster mean")
plt.title("Network topology robustness: cluster count")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_topology_cluster_mean_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["switching_rate_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Network topology robustness: switching rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_topology_switching_rate_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for topology in TOPOLOGIES:
    sub = agg[agg["topology"] == topology]
    plt.plot(sub["K"], sub["reorganization_score_mean"], marker="o", label=topology)
plt.xlabel("K")
plt.ylabel("Reorganization score")
plt.title("Network topology robustness: reorganization score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_topology_reorganization_score.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.scatter(summary["peak_K_reorganization"], summary["topology"], s=120)
plt.xlabel("Peak K")
plt.ylabel("Topology")
plt.title("Peak reorganization K across network topologies")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_topology_peak_K_summary.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
import os
import shutil

SOURCE = "notebooks/Kuramoto/stuart_landau_network_topology_v1_CHECKPOINT"
TARGET = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE/stuart_landau_network_topology_v1"

os.makedirs(TARGET, exist_ok=True)

for f in os.listdir(SOURCE):
    src = os.path.join(SOURCE, f)
    dst = os.path.join(TARGET, f)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("DONE")
print(TARGET)

In [ ]:
# =========================
# REBUILD MASTER INDEX AFTER NETWORK TOPOLOGY
# =========================

import os
import pandas as pd
from pathlib import Path

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

rows = []

for path, dirs, files in os.walk(ROOT):
    for f in files:
        if f.endswith((".csv", ".png", ".docx", ".pdf")):
            full = os.path.join(path, f)
            rel = os.path.relpath(full, ROOT)

            parts = Path(rel).parts
            source = parts[0] if len(parts) > 0 else ""
            folder = parts[-2] if len(parts) > 1 else ""
            ext = Path(f).suffix.lower()

            rows.append({
                "source": source,
                "folder": folder,
                "filename": f,
                "extension": ext,
                "size_kb": round(os.path.getsize(full) / 1024, 2),
                "path": full
            })

index_df = pd.DataFrame(rows)
index_df = index_df.sort_values(["folder", "extension", "filename"]).reset_index(drop=True)

index_path = ROOT + "/MASTER_INDEX.csv"
index_df.to_csv(index_path, index=False)

summary = (
    index_df
    .groupby(["folder", "extension"])
    .size()
    .reset_index(name="file_count")
    .sort_values(["folder", "extension"])
)

summary_path = ROOT + "/MASTER_INDEX_SUMMARY.csv"
summary.to_csv(summary_path, index=False)

print("MASTER_INDEX saved:", index_path)
print("MASTER_INDEX_SUMMARY saved:", summary_path)
print("Total files:", len(index_df))

display(summary)

In [ ]:
# ============================================================
# STUART-LANDAU IRREVERSIBILITY THRESHOLD v1 — CHECKPOINT
# reversible / metastable / irreversible transition after perturbation
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

# ============================================================
# OUTPUT
# ============================================================

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/stuart_landau_irreversibility_threshold_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/irreversibility_threshold_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))

K_VALUES = np.round(np.arange(0.4, 1.201, 0.05), 3)
KICK_VALUES = np.round(np.arange(0.0, 1.01, 0.1), 2)

omega_std = 0.4

dt = 0.03
pre_steps = 2500
post_steps = 2500
discard_pre = 1000
discard_post = 1000

alpha = 1.0
cluster_threshold = 0.15

# memory/core definition
core_fraction = 0.15

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def circular_similarity(theta_a, theta_b):
    return float(np.abs(np.mean(np.exp(1j * (theta_a - theta_b)))))

def phase_dispersion(theta):
    return float(1 - np.abs(np.mean(np.exp(1j * theta))))

def run_stuart_landau(z, omega, K, steps, discard):
    R_series = []
    cluster_series = []
    theta_series = []

    for t in range(steps):
        mean_z = np.mean(z)

        dz = (
            (alpha + 1j * omega - np.abs(z)**2) * z
            + K * (mean_z - z)
        )

        z = z + dt * dz

        if t >= discard:
            theta = np.angle(z)
            R_series.append(order_parameter(theta))
            cluster_series.append(count_clusters(theta, cluster_threshold))
            theta_series.append(theta.copy())

    return z, np.array(R_series), np.array(cluster_series), np.array(theta_series)

def identify_core(theta_series, fraction=0.15):
    # core = oscillators with smallest phase variance over stabilized window
    phases = np.unwrap(theta_series, axis=0)
    phase_var = np.var(phases, axis=0)

    n_core = int(len(phase_var) * fraction)
    core_idx = np.argsort(phase_var)[:n_core]

    return core_idx, phase_var

def kick_system(z, kick_strength, rng, mode="phase"):
    theta = np.angle(z)
    amp = np.abs(z)

    if mode == "phase":
        theta_new = theta + rng.normal(0, kick_strength, len(theta))
        return amp * np.exp(1j * theta_new)

    if mode == "amplitude":
        amp_new = amp * (1 + rng.normal(0, kick_strength, len(amp)))
        amp_new = np.clip(amp_new, 0.01, None)
        return amp_new * np.exp(1j * theta)

    if mode == "mixed":
        theta_new = theta + rng.normal(0, kick_strength, len(theta))
        amp_new = amp * (1 + rng.normal(0, kick_strength, len(amp)))
        amp_new = np.clip(amp_new, 0.01, None)
        return amp_new * np.exp(1j * theta_new)

    raise ValueError("Unknown kick mode")

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(
        all_df["K"].round(3),
        all_df["kick_strength"].round(2),
        all_df["seed"]
    ))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for K in K_VALUES:

    print("\n" + "="*70)
    print("K =", K)
    print("="*70)

    for kick_strength in KICK_VALUES:

        print(f"\nKick strength = {kick_strength}")

        for seed in tqdm(SEEDS):

            key = (round(float(K), 3), round(float(kick_strength), 2), int(seed))

            if key in done:
                continue

            rng = np.random.default_rng(seed)

            omega = rng.normal(0, omega_std, N)

            z0 = (
                rng.normal(0, 0.5, N)
                + 1j * rng.normal(0, 0.5, N)
            )

            # ----------------------------
            # PRE-STABILIZATION
            # ----------------------------

            z_pre, R_pre, clusters_pre, theta_pre_series = run_stuart_landau(
                z0.copy(), omega, K, pre_steps, discard_pre
            )

            theta_pre_final = np.angle(z_pre)
            R_pre_mean = float(np.mean(R_pre))
            cluster_pre_mean = float(np.mean(clusters_pre))
            cluster_pre_mode = int(pd.Series(clusters_pre).mode().iloc[0])

            core_pre_idx, phase_var_pre = identify_core(theta_pre_series, core_fraction)

            core_pre_set = set(core_pre_idx.tolist())

            # ----------------------------
            # KICK
            # ----------------------------

            z_kicked = kick_system(z_pre.copy(), kick_strength, rng, mode="mixed")

            theta_after_kick = np.angle(z_kicked)

            kick_distance = 1 - circular_similarity(theta_pre_final, theta_after_kick)

            # ----------------------------
            # POST-RELAXATION
            # ----------------------------

            z_post, R_post, clusters_post, theta_post_series = run_stuart_landau(
                z_kicked.copy(), omega, K, post_steps, discard_post
            )

            theta_post_final = np.angle(z_post)

            R_post_mean = float(np.mean(R_post))
            cluster_post_mean = float(np.mean(clusters_post))
            cluster_post_mode = int(pd.Series(clusters_post).mode().iloc[0])

            core_post_idx, phase_var_post = identify_core(theta_post_series, core_fraction)
            core_post_set = set(core_post_idx.tolist())

            # ----------------------------
            # METRICS
            # ----------------------------

            recovery_similarity = circular_similarity(theta_pre_final, theta_post_final)

            memory_loss = 1 - recovery_similarity

            core_overlap = len(core_pre_set.intersection(core_post_set)) / max(1, len(core_pre_set))

            new_core_fraction = 1 - core_overlap

            cluster_shift = abs(cluster_post_mode - cluster_pre_mode)

            R_change = abs(R_post_mean - R_pre_mean)

            cluster_change = abs(cluster_post_mean - cluster_pre_mean)

            # high irreversibility = poor recovery + changed core + changed clustering
            irreversibility_score = (
                memory_loss
                + new_core_fraction
                + 0.25 * cluster_change
                + 0.25 * R_change
            )

            # regime classification
            if recovery_similarity > 0.85 and core_overlap > 0.75:
                regime = "reversible"
            elif recovery_similarity > 0.55 and core_overlap > 0.40:
                regime = "metastable"
            else:
                regime = "irreversible"

            row = {
                "K": K,
                "kick_strength": kick_strength,
                "seed": seed,

                "R_pre_mean": R_pre_mean,
                "R_post_mean": R_post_mean,
                "R_change": R_change,

                "cluster_pre_mean": cluster_pre_mean,
                "cluster_post_mean": cluster_post_mean,
                "cluster_change": cluster_change,

                "cluster_pre_mode": cluster_pre_mode,
                "cluster_post_mode": cluster_post_mode,
                "cluster_shift": cluster_shift,

                "kick_distance": kick_distance,
                "recovery_similarity": recovery_similarity,
                "memory_loss": memory_loss,

                "core_overlap": core_overlap,
                "new_core_fraction": new_core_fraction,

                "irreversibility_score": irreversibility_score,
                "regime": regime
            }

            rows.append(row)
            done.add(key)

            pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# DATAFRAMES
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby(["K", "kick_strength"])
    .agg({
        "R_pre_mean": ["mean", "std"],
        "R_post_mean": ["mean", "std"],
        "R_change": ["mean", "std"],
        "cluster_change": ["mean", "std"],
        "kick_distance": ["mean", "std"],
        "recovery_similarity": ["mean", "std"],
        "memory_loss": ["mean", "std"],
        "core_overlap": ["mean", "std"],
        "new_core_fraction": ["mean", "std"],
        "irreversibility_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

# regime fractions
regime_counts = (
    all_df
    .groupby(["K", "kick_strength", "regime"])
    .size()
    .reset_index(name="count")
)

regime_pivot = regime_counts.pivot_table(
    index=["K", "kick_strength"],
    columns="regime",
    values="count",
    fill_value=0
).reset_index()

for col in ["reversible", "metastable", "irreversible"]:
    if col not in regime_pivot.columns:
        regime_pivot[col] = 0

regime_pivot["total"] = (
    regime_pivot["reversible"]
    + regime_pivot["metastable"]
    + regime_pivot["irreversible"]
)

regime_pivot["reversible_fraction"] = regime_pivot["reversible"] / regime_pivot["total"]
regime_pivot["metastable_fraction"] = regime_pivot["metastable"] / regime_pivot["total"]
regime_pivot["irreversible_fraction"] = regime_pivot["irreversible"] / regime_pivot["total"]

agg = agg.merge(
    regime_pivot[
        [
            "K", "kick_strength",
            "reversible_fraction",
            "metastable_fraction",
            "irreversible_fraction"
        ]
    ],
    on=["K", "kick_strength"],
    how="left"
)

# ============================================================
# SUMMARY
# ============================================================

peak_idx = agg["irreversibility_score_mean"].idxmax()

# threshold: first kick where irreversible_fraction > 0.5 for each K
threshold_rows = []

for K in K_VALUES:
    sub = agg[agg["K"] == K].sort_values("kick_strength")
    hit = sub[sub["irreversible_fraction"] > 0.5]

    if len(hit) > 0:
        threshold = hit.iloc[0]["kick_strength"]
    else:
        threshold = np.nan

    threshold_rows.append({
        "K": K,
        "irreversibility_threshold_kick": threshold
    })

threshold_df = pd.DataFrame(threshold_rows)

summary = pd.DataFrame([{
    "peak_K_irreversibility": agg.loc[peak_idx, "K"],
    "peak_kick_strength_irreversibility": agg.loc[peak_idx, "kick_strength"],
    "peak_irreversibility_score": agg.loc[peak_idx, "irreversibility_score_mean"],
    "recovery_similarity_at_peak": agg.loc[peak_idx, "recovery_similarity_mean"],
    "core_overlap_at_peak": agg.loc[peak_idx, "core_overlap_mean"],
    "irreversible_fraction_at_peak": agg.loc[peak_idx, "irreversible_fraction"]
}])

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/irreversibility_threshold_all.csv", index=False)
agg.to_csv(OUT + "/irreversibility_threshold_aggregate.csv", index=False)
summary.to_csv(OUT + "/irreversibility_threshold_summary.csv", index=False)
threshold_df.to_csv(OUT + "/irreversibility_threshold_by_K.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

# Heatmap helper
def pivot_metric(df, metric):
    return df.pivot(index="K", columns="kick_strength", values=metric)

# 1. Irreversibility heatmap
heat = pivot_metric(agg, "irreversibility_score_mean")

plt.figure(figsize=(10,6))
plt.imshow(
    heat.values,
    aspect="auto",
    origin="lower",
    extent=[
        KICK_VALUES.min(), KICK_VALUES.max(),
        K_VALUES.min(), K_VALUES.max()
    ]
)
plt.colorbar(label="Irreversibility score")
plt.xlabel("Kick strength")
plt.ylabel("K")
plt.title("Irreversibility threshold map")
plt.tight_layout()
plt.savefig(OUT + "/fig_irreversibility_heatmap.png", dpi=300)
plt.show()

# 2. Recovery similarity heatmap
heat = pivot_metric(agg, "recovery_similarity_mean")

plt.figure(figsize=(10,6))
plt.imshow(
    heat.values,
    aspect="auto",
    origin="lower",
    extent=[
        KICK_VALUES.min(), KICK_VALUES.max(),
        K_VALUES.min(), K_VALUES.max()
    ]
)
plt.colorbar(label="Recovery similarity")
plt.xlabel("Kick strength")
plt.ylabel("K")
plt.title("Recovery similarity after perturbation")
plt.tight_layout()
plt.savefig(OUT + "/fig_recovery_similarity_heatmap.png", dpi=300)
plt.show()

# 3. Core overlap heatmap
heat = pivot_metric(agg, "core_overlap_mean")

plt.figure(figsize=(10,6))
plt.imshow(
    heat.values,
    aspect="auto",
    origin="lower",
    extent=[
        KICK_VALUES.min(), KICK_VALUES.max(),
        K_VALUES.min(), K_VALUES.max()
    ]
)
plt.colorbar(label="Core overlap")
plt.xlabel("Kick strength")
plt.ylabel("K")
plt.title("Core persistence after perturbation")
plt.tight_layout()
plt.savefig(OUT + "/fig_core_overlap_heatmap.png", dpi=300)
plt.show()

# 4. Threshold curve
plt.figure(figsize=(9,5))
plt.plot(
    threshold_df["K"],
    threshold_df["irreversibility_threshold_kick"],
    marker="o"
)
plt.xlabel("K")
plt.ylabel("Kick threshold for irreversible fraction > 0.5")
plt.title("Irreversibility threshold by coupling K")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_irreversibility_threshold_by_K.png", dpi=300)
plt.show()

# 5. Regime fractions for selected K
selected_K = [0.5, 0.6, 0.7, 0.8, 1.0]

plt.figure(figsize=(10,6))

for K in selected_K:
    sub = agg[agg["K"] == K]
    if len(sub) > 0:
        plt.plot(
            sub["kick_strength"],
            sub["irreversible_fraction"],
            marker="o",
            label=f"K={K}"
        )

plt.xlabel("Kick strength")
plt.ylabel("Irreversible fraction")
plt.title("Irreversibility fraction vs perturbation strength")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_irreversible_fraction_selected_K.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nTHRESHOLD BY K")
display(threshold_df)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
import pandas as pd
import os

OUT = "notebooks/Kuramoto/stuart_landau_irreversibility_threshold_v1_CHECKPOINT"

all_df = pd.read_csv(OUT + "/irreversibility_threshold_all.csv")

zero = all_df[all_df["kick_strength"] == 0.0].copy()

diag = (
    zero.groupby("K")
    .agg({
        "recovery_similarity": ["mean", "std", "min", "max"],
        "core_overlap": ["mean", "std", "min", "max"],
        "memory_loss": ["mean", "std"],
        "irreversibility_score": ["mean", "std"],
        "R_change": ["mean", "std"],
        "cluster_change": ["mean", "std"],
    })
)

diag.columns = ["_".join(c) for c in diag.columns]
diag = diag.reset_index()

diag.to_csv(OUT + "/kick_zero_diagnostic.csv", index=False)

print("Saved kick-zero diagnostic")
display(diag)

In [ ]:
import pandas as pd
import numpy as np
import os

OUT = "notebooks/Kuramoto/stuart_landau_irreversibility_threshold_v1_CHECKPOINT"

agg = pd.read_csv(OUT + "/irreversibility_threshold_aggregate.csv")

# baseline at kick=0 for each K
baseline = (
    agg[agg["kick_strength"] == 0.0]
    [["K", "irreversibility_score_mean", "recovery_similarity_mean", "memory_loss_mean",
      "R_change_mean", "cluster_change_mean", "irreversible_fraction"]]
    .rename(columns={
        "irreversibility_score_mean": "baseline_irreversibility_score",
        "recovery_similarity_mean": "baseline_recovery_similarity",
        "memory_loss_mean": "baseline_memory_loss",
        "R_change_mean": "baseline_R_change",
        "cluster_change_mean": "baseline_cluster_change",
        "irreversible_fraction": "baseline_irreversible_fraction"
    })
)

corrected = agg.merge(baseline, on="K", how="left")

# baseline corrected metrics
corrected["excess_irreversibility_score"] = (
    corrected["irreversibility_score_mean"]
    - corrected["baseline_irreversibility_score"]
)

corrected["excess_memory_loss"] = (
    corrected["memory_loss_mean"]
    - corrected["baseline_memory_loss"]
)

corrected["excess_R_change"] = (
    corrected["R_change_mean"]
    - corrected["baseline_R_change"]
)

corrected["excess_cluster_change"] = (
    corrected["cluster_change_mean"]
    - corrected["baseline_cluster_change"]
)

corrected["excess_irreversible_fraction"] = (
    corrected["irreversible_fraction"]
    - corrected["baseline_irreversible_fraction"]
)

# global-state irreversibility: does not use core_overlap
corrected["global_state_irreversibility"] = (
    corrected["memory_loss_mean"]
    + 0.5 * corrected["R_change_mean"]
    + 0.25 * corrected["cluster_change_mean"]
)

baseline_global = (
    corrected[corrected["kick_strength"] == 0.0]
    [["K", "global_state_irreversibility"]]
    .rename(columns={"global_state_irreversibility": "baseline_global_state_irreversibility"})
)

corrected = corrected.merge(baseline_global, on="K", how="left")

corrected["excess_global_state_irreversibility"] = (
    corrected["global_state_irreversibility"]
    - corrected["baseline_global_state_irreversibility"]
)

# summary: peak excess global irreversibility
peak_idx = corrected["excess_global_state_irreversibility"].idxmax()

summary_corrected = pd.DataFrame([{
    "peak_K_excess_global_irreversibility": corrected.loc[peak_idx, "K"],
    "peak_kick_strength_excess_global_irreversibility": corrected.loc[peak_idx, "kick_strength"],
    "peak_excess_global_state_irreversibility": corrected.loc[peak_idx, "excess_global_state_irreversibility"],
    "raw_global_state_irreversibility_at_peak": corrected.loc[peak_idx, "global_state_irreversibility"],
    "baseline_global_state_irreversibility_at_peak_K": corrected.loc[peak_idx, "baseline_global_state_irreversibility"],
    "recovery_similarity_at_peak": corrected.loc[peak_idx, "recovery_similarity_mean"],
    "R_change_at_peak": corrected.loc[peak_idx, "R_change_mean"],
    "cluster_change_at_peak": corrected.loc[peak_idx, "cluster_change_mean"]
}])

# threshold: pierwszy kick gdzie excess_global_state_irreversibility > 0.1
threshold_rows = []

for K in sorted(corrected["K"].unique()):
    sub = corrected[corrected["K"] == K].sort_values("kick_strength")
    hit = sub[sub["excess_global_state_irreversibility"] > 0.1]
    threshold = hit.iloc[0]["kick_strength"] if len(hit) else np.nan

    threshold_rows.append({
        "K": K,
        "excess_global_irreversibility_threshold_kick": threshold
    })

threshold_corrected = pd.DataFrame(threshold_rows)

corrected.to_csv(OUT + "/irreversibility_threshold_aggregate_global_corrected.csv", index=False)
summary_corrected.to_csv(OUT + "/irreversibility_threshold_summary_global_corrected.csv", index=False)
threshold_corrected.to_csv(OUT + "/irreversibility_threshold_by_K_global_corrected.csv", index=False)

print("GLOBAL-CORRECTED SUMMARY")
display(summary_corrected)

print("GLOBAL-CORRECTED THRESHOLD BY K")
display(threshold_corrected)

In [ ]:
import os
import shutil

SOURCE = "notebooks/Kuramoto/stuart_landau_irreversibility_threshold_v1_CHECKPOINT"
TARGET = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE/stuart_landau_irreversibility_threshold_v1"

os.makedirs(TARGET, exist_ok=True)

for f in os.listdir(SOURCE):
    src = os.path.join(SOURCE, f)
    dst = os.path.join(TARGET, f)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("DONE")
print(TARGET)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import networkx as nx
import os

SAVE_DIR = "notebooks/Kuramoto/stuart_landau_metastability_landscape_v1"
os.makedirs(SAVE_DIR, exist_ok=True)

N = 60
dt = 0.03
steps = 2400
discard = 1200

K_values_forward = np.linspace(0.2, 1.2, 21)
K_values_backward = K_values_forward[::-1]

n_runs = 20

alpha = 1.0
omega_sigma = 0.15

rng = np.random.default_rng(42)

A = nx.watts_strogatz_graph(N, 6, 0.12, seed=42)
A = nx.to_numpy_array(A)

def simulate(K, z0=None):

    omega = rng.normal(0, omega_sigma, N)

    if z0 is None:
        z = (
            rng.normal(0, 0.5, N)
            + 1j * rng.normal(0, 0.5, N)
        )
    else:
        z = z0.copy()

    R_series = []

    for t in range(steps):

        coupling = A @ z - np.sum(A, axis=1) * z

        dz = (
            (alpha + 1j * omega - np.abs(z)**2) * z
            + K * coupling
        )

        z += dt * dz

        if t >= discard:
            phases = np.angle(z)
            R = np.abs(np.mean(np.exp(1j * phases)))
            R_series.append(R)

    return z, np.mean(R_series), np.std(R_series)

forward_results = []
backward_results = []

for run in tqdm(range(n_runs)):

    z = None

    for K in K_values_forward:

        z, Rm, Rs = simulate(K, z)

        forward_results.append({
            "run": run,
            "direction": "forward",
            "K": K,
            "R_mean": Rm,
            "R_std": Rs
        })

    for K in K_values_backward:

        z, Rm, Rs = simulate(K, z)

        backward_results.append({
            "run": run,
            "direction": "backward",
            "K": K,
            "R_mean": Rm,
            "R_std": Rs
        })

df_forward = pd.DataFrame(forward_results)
df_backward = pd.DataFrame(backward_results)

df = pd.concat([df_forward, df_backward], ignore_index=True)

agg = (
    df.groupby(["direction", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"]
    })
)

agg.columns = [
    "_".join(col).strip()
    for col in agg.columns.values
]

agg = agg.reset_index()

forward_curve = agg[agg["direction"] == "forward"]
backward_curve = agg[agg["direction"] == "backward"]

hysteresis_gap = np.mean(
    np.abs(
        forward_curve["R_mean_mean"].values
        - backward_curve["R_mean_mean"].values
    )
)

summary = pd.DataFrame([{
    "mean_hysteresis_gap": hysteresis_gap,
    "max_forward_R": forward_curve["R_mean_mean"].max(),
    "max_backward_R": backward_curve["R_mean_mean"].max()
}])

df.to_csv(
    os.path.join(SAVE_DIR, "metastability_landscape_raw.csv"),
    index=False
)

agg.to_csv(
    os.path.join(SAVE_DIR, "metastability_landscape_aggregate.csv"),
    index=False
)

summary.to_csv(
    os.path.join(SAVE_DIR, "metastability_landscape_summary.csv"),
    index=False
)

plt.figure(figsize=(8,5))

plt.plot(
    forward_curve["K"],
    forward_curve["R_mean_mean"],
    label="Forward"
)

plt.plot(
    backward_curve["K"],
    backward_curve["R_mean_mean"],
    label="Backward"
)

plt.xlabel("K")
plt.ylabel("Mean Synchronization R")
plt.title("Metastability / Hysteresis Landscape")
plt.legend()

plt.tight_layout()

plot_path = os.path.join(
    SAVE_DIR,
    "metastability_landscape_plot.png"
)

plt.savefig(plot_path, dpi=300)
plt.close()

print("\nSUMMARY\n")
print(summary)

print("\nSaved to:")
print(SAVE_DIR)

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import networkx as nx
import os

SAVE_DIR = "notebooks/Kuramoto/stuart_landau_outsider_recruitment_v1"
os.makedirs(SAVE_DIR, exist_ok=True)

N_core = 50
N_outsiders = 10
N = N_core + N_outsiders

dt = 0.03
steps = 2600
discard = 1200

alpha = 1.0
omega_sigma_core = 0.12
omega_sigma_out = 1.4

K_values = np.linspace(0.3, 1.2, 19)

n_runs = 20

rng = np.random.default_rng(42)

G = nx.watts_strogatz_graph(N, 6, 0.12, seed=42)
A = nx.to_numpy_array(G)

outsider_idx = np.arange(N_core, N)

results = []

def simulate(K):

    omega = np.concatenate([
        rng.normal(0, omega_sigma_core, N_core),
        rng.normal(0, omega_sigma_out, N_outsiders)
    ])

    z = (
        rng.normal(0, 0.5, N)
        + 1j * rng.normal(0, 0.5, N)
    )

    outsider_sync_series = []
    global_R_series = []

    for t in range(steps):

        coupling = A @ z - np.sum(A, axis=1) * z

        dz = (
            (alpha + 1j * omega - np.abs(z)**2) * z
            + K * coupling
        )

        z += dt * dz

        if t >= discard:

            phases = np.angle(z)

            global_R = np.abs(
                np.mean(np.exp(1j * phases))
            )

            outsider_R = np.abs(
                np.mean(
                    np.exp(1j * phases[outsider_idx])
                )
            )

            outsider_sync_series.append(outsider_R)
            global_R_series.append(global_R)

    return (
        np.mean(global_R_series),
        np.mean(outsider_sync_series),
        np.std(outsider_sync_series)
    )

for K in tqdm(K_values):

    for run in range(n_runs):

        Rg, Rout, Rout_std = simulate(K)

        outsider_recruitment = Rout / (Rg + 1e-12)

        results.append({
            "K": K,
            "run": run,
            "global_R": Rg,
            "outsider_R": Rout,
            "outsider_R_std": Rout_std,
            "outsider_recruitment_score": outsider_recruitment
        })

df = pd.DataFrame(results)

agg = (
    df.groupby("K")
    .agg({
        "global_R": ["mean", "std"],
        "outsider_R": ["mean", "std"],
        "outsider_recruitment_score": ["mean", "std"]
    })
)

agg.columns = [
    "_".join(col).strip()
    for col in agg.columns.values
]

agg = agg.reset_index()

peak_idx = agg["outsider_recruitment_score_mean"].idxmax()

summary = pd.DataFrame([{
    "peak_K_outsider_recruitment":
        agg.loc[peak_idx, "K"],

    "peak_outsider_recruitment_score":
        agg.loc[peak_idx,
                "outsider_recruitment_score_mean"],

    "global_R_at_peak":
        agg.loc[peak_idx, "global_R_mean"],

    "outsider_R_at_peak":
        agg.loc[peak_idx, "outsider_R_mean"]
}])

df.to_csv(
    os.path.join(SAVE_DIR,
                 "outsider_recruitment_raw.csv"),
    index=False
)

agg.to_csv(
    os.path.join(SAVE_DIR,
                 "outsider_recruitment_aggregate.csv"),
    index=False
)

summary.to_csv(
    os.path.join(SAVE_DIR,
                 "outsider_recruitment_summary.csv"),
    index=False
)

plt.figure(figsize=(8,5))

plt.plot(
    agg["K"],
    agg["outsider_recruitment_score_mean"]
)

plt.xlabel("K")
plt.ylabel("Outsider Recruitment Score")
plt.title("Dynamic Outsider Recruitment")

plt.tight_layout()

plot_path = os.path.join(
    SAVE_DIR,
    "outsider_recruitment_plot.png"
)

plt.savefig(plot_path, dpi=300)
plt.close()

print("\nSUMMARY\n")
print(summary)

print("\nSaved to:")
print(SAVE_DIR)

In [ ]:
import os
import shutil
import pandas as pd

MASTER_DIR = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

NEW_FOLDERS = [
    "notebooks/Kuramoto/stuart_landau_metastability_landscape_v1",
    "notebooks/Kuramoto/stuart_landau_outsider_recruitment_v1"
]

for src_folder in NEW_FOLDERS:

    folder_name = os.path.basename(src_folder)
    dst_folder = os.path.join(MASTER_DIR, folder_name)

    os.makedirs(dst_folder, exist_ok=True)

    for f in os.listdir(src_folder):

        src = os.path.join(src_folder, f)
        dst = os.path.join(dst_folder, f)

        if os.path.isfile(src):
            shutil.copy2(src, dst)

records = []

for root, dirs, files in os.walk(MASTER_DIR):

    for f in files:

        ext = os.path.splitext(f)[1]

        records.append({
            "folder": os.path.basename(root),
            "file": f,
            "extension": ext
        })

master_df = pd.DataFrame(records)

master_df.to_csv(
    os.path.join(MASTER_DIR, "MASTER_INDEX.csv"),
    index=False
)

summary = (
    master_df.groupby(["folder", "extension"])
    .size()
    .reset_index(name="file_count")
)

summary.to_csv(
    os.path.join(MASTER_DIR, "MASTER_INDEX_SUMMARY.csv"),
    index=False
)

print("MASTER_INDEX updated")
print("Total files:", len(master_df))

display(summary.sort_values(["folder", "extension"]))

In [ ]:
import os
import pandas as pd
from glob import glob

MASTER_DIR = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

summary_files = glob(MASTER_DIR + "/**/*summary*.csv", recursive=True)

rows = []

for f in summary_files:
    try:
        df = pd.read_csv(f)
        row = {
            "folder": os.path.basename(os.path.dirname(f)),
            "summary_file": os.path.basename(f),
            "path": f
        }

        if len(df) > 0:
            for col in df.columns:
                row[col] = df.iloc[0][col]

        rows.append(row)

    except Exception as e:
        rows.append({
            "folder": os.path.basename(os.path.dirname(f)),
            "summary_file": os.path.basename(f),
            "error": str(e),
            "path": f
        })

master_results = pd.DataFrame(rows)

out = os.path.join(MASTER_DIR, "MASTER_RESULTS_SUMMARY.csv")
master_results.to_csv(out, index=False)

print("Saved:", out)
print("Summary files:", len(master_results))

display(master_results)

In [ ]:
# ============================================================
# STUART-LANDAU NULL / CONTROL TEST v1 — CHECKPOINT
# controls: original vs shuffled omega vs random phase reset
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/stuart_landau_null_control_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/null_control_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

N = 400
SEEDS = list(range(20))
K_VALUES = np.round(np.arange(0.4, 1.201, 0.05), 3)

omega_std = 0.4

dt = 0.03
steps = 3000
discard = 1000

alpha = 1.0
cluster_threshold = 0.15

CONTROL_TYPES = [
    "original",
    "omega_shuffled",
    "phase_reset"
]

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def memory_score(cluster_series, window=200):
    s = pd.Series(cluster_series)
    return float((s.rolling(window).std().fillna(0) < 0.2).mean())

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(
        all_df["control_type"],
        all_df["K"].round(3),
        all_df["seed"]
    ))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for control_type in CONTROL_TYPES:

    print("\n" + "="*70)
    print("CONTROL:", control_type)
    print("="*70)

    for K in K_VALUES:

        print(f"\nK={K}")

        for seed in tqdm(SEEDS):

            key = (control_type, round(float(K), 3), int(seed))

            if key in done:
                continue

            rng = np.random.default_rng(seed)

            omega = rng.normal(0, omega_std, N)

            z = (
                rng.normal(0, 0.5, N)
                + 1j * rng.normal(0, 0.5, N)
            )

            if control_type == "omega_shuffled":
                rng.shuffle(omega)

            if control_type == "phase_reset":
                amp = np.abs(z)
                theta = rng.uniform(-np.pi, np.pi, N)
                z = amp * np.exp(1j * theta)

            R_series = []
            cluster_series = []

            for t in range(steps):

                mean_z = np.mean(z)

                dz = (
                    (alpha + 1j * omega - np.abs(z)**2) * z
                    + K * (mean_z - z)
                )

                z = z + dt * dz

                if t >= discard:
                    theta = np.angle(z)
                    R_series.append(order_parameter(theta))
                    cluster_series.append(count_clusters(theta, cluster_threshold))

            R_series = np.array(R_series)
            cluster_series = np.array(cluster_series)

            dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
            dominant_fraction = float(np.mean(cluster_series == dominant_cluster))

            switching_rate = float(np.sum(np.diff(cluster_series) != 0) / len(cluster_series))

            reorganization_score = float(
                np.std(cluster_series)
                * (1 - dominant_fraction)
                * (switching_rate + 1e-9)
            )

            row = {
                "control_type": control_type,
                "K": K,
                "seed": seed,

                "R_mean": float(np.mean(R_series)),
                "R_std": float(np.std(R_series)),

                "cluster_mean": float(np.mean(cluster_series)),
                "cluster_std": float(np.std(cluster_series)),

                "dominant_cluster": int(dominant_cluster),
                "dominant_fraction": dominant_fraction,
                "switching_rate": switching_rate,
                "memory_score": memory_score(cluster_series),

                "reorganization_score": reorganization_score
            }

            rows.append(row)
            done.add(key)

            pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby(["control_type", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_score": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

summary_rows = []

for control_type in CONTROL_TYPES:
    sub = agg[agg["control_type"] == control_type]

    idx = sub["reorganization_score_mean"].idxmax()

    summary_rows.append({
        "control_type": control_type,
        "peak_K_reorganization": sub.loc[idx, "K"],
        "peak_reorganization_score": sub.loc[idx, "reorganization_score_mean"],
        "R_mean_at_peak": sub.loc[idx, "R_mean_mean"],
        "cluster_mean_at_peak": sub.loc[idx, "cluster_mean_mean"],
        "dominant_fraction_at_peak": sub.loc[idx, "dominant_fraction_mean"],
        "switching_rate_at_peak": sub.loc[idx, "switching_rate_mean"],
        "memory_score_at_peak": sub.loc[idx, "memory_score_mean"],
    })

summary = pd.DataFrame(summary_rows)

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/null_control_all.csv", index=False)
agg.to_csv(OUT + "/null_control_aggregate.csv", index=False)
summary.to_csv(OUT + "/null_control_summary.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(10,6))
for control_type in CONTROL_TYPES:
    sub = agg[agg["control_type"] == control_type]
    plt.plot(sub["K"], sub["reorganization_score_mean"], marker="o", label=control_type)
plt.xlabel("K")
plt.ylabel("Reorganization score")
plt.title("Null/control test: reorganization score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_null_control_reorganization_score.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for control_type in CONTROL_TYPES:
    sub = agg[agg["control_type"] == control_type]
    plt.plot(sub["K"], sub["switching_rate_mean"], marker="o", label=control_type)
plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Null/control test: switching rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_null_control_switching_rate.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for control_type in CONTROL_TYPES:
    sub = agg[agg["control_type"] == control_type]
    plt.plot(sub["K"], sub["memory_score_mean"], marker="o", label=control_type)
plt.xlabel("K")
plt.ylabel("Memory score")
plt.title("Null/control test: memory score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_null_control_memory_score.png", dpi=300)
plt.show()

plt.figure(figsize=(9,5))
plt.scatter(summary["peak_K_reorganization"], summary["control_type"], s=120)
plt.xlabel("Peak K")
plt.ylabel("Control type")
plt.title("Peak reorganization K across controls")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_null_control_peak_K_summary.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
import os
import shutil

SOURCE = "notebooks/Kuramoto/stuart_landau_null_control_v1_CHECKPOINT"
TARGET = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE/stuart_landau_null_control_v1"

os.makedirs(TARGET, exist_ok=True)

for f in os.listdir(SOURCE):
    src = os.path.join(SOURCE, f)
    dst = os.path.join(TARGET, f)

    if os.path.isfile(src):
        shutil.copy2(src, dst)

print("DONE")
print(TARGET)

In [ ]:
import os
import pandas as pd
from pathlib import Path

ROOT = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

rows = []

for path, dirs, files in os.walk(ROOT):
    for f in files:
        if f.endswith((".csv", ".png", ".docx", ".pdf")):
            full = os.path.join(path, f)
            rel = os.path.relpath(full, ROOT)
            parts = Path(rel).parts

            rows.append({
                "source": parts[0] if len(parts) > 0 else "",
                "folder": parts[-2] if len(parts) > 1 else "",
                "filename": f,
                "extension": Path(f).suffix.lower(),
                "size_kb": round(os.path.getsize(full) / 1024, 2),
                "path": full
            })

index_df = pd.DataFrame(rows).sort_values(
    ["folder", "extension", "filename"]
).reset_index(drop=True)

index_path = ROOT + "/MASTER_INDEX.csv"
index_df.to_csv(index_path, index=False)

summary = (
    index_df.groupby(["folder", "extension"])
    .size()
    .reset_index(name="file_count")
    .sort_values(["folder", "extension"])
)

summary_path = ROOT + "/MASTER_INDEX_SUMMARY.csv"
summary.to_csv(summary_path, index=False)

print("MASTER_INDEX saved:", index_path)
print("MASTER_INDEX_SUMMARY saved:", summary_path)
print("Total files:", len(index_df))

display(summary)

In [ ]:
# ============================================================
# STUART-LANDAU UNIFIED PHASE DIAGRAM / FINAL TECH SUMMARY
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

MASTER_DIR = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"
OUT = os.path.join(MASTER_DIR, "stuart_landau_unified_phase_diagram")

os.makedirs(OUT, exist_ok=True)

# ------------------------------------------------------------
# Collect all summary files
# ------------------------------------------------------------

summary_files = glob(MASTER_DIR + "/**/*summary*.csv", recursive=True)

records = []

for f in summary_files:
    folder = os.path.basename(os.path.dirname(f))
    name = os.path.basename(f)

    try:
        df = pd.read_csv(f)

        if len(df) == 0:
            continue

        row = df.iloc[0].to_dict()
        row["folder"] = folder
        row["summary_file"] = name
        row["path"] = f

        records.append(row)

    except Exception as e:
        records.append({
            "folder": folder,
            "summary_file": name,
            "error": str(e),
            "path": f
        })

raw_summary = pd.DataFrame(records)

raw_summary.to_csv(
    os.path.join(OUT, "unified_raw_summary_collection.csv"),
    index=False
)

# ------------------------------------------------------------
# Manual standardized peak extraction
# ------------------------------------------------------------

rows = []

def add(test, K, metric, value, category, source):
    rows.append({
        "test": test,
        "K_peak_or_threshold": K,
        "main_metric": metric,
        "metric_value": value,
        "category": category,
        "source_file": source
    })

for _, r in raw_summary.iterrows():

    folder = r.get("folder", "")
    source = r.get("summary_file", "")

    # Critical slowing
    if folder == "stuart_landau_critical_slowing_output":
        add(
            "Critical slowing",
            r.get("peak_K_critical_slowing", np.nan),
            "critical slowing score",
            r.get("peak_critical_slowing_score", np.nan),
            "early warning",
            source
        )

    # Basin memory
    elif folder == "stuart_landau_basin_memory_output":
        add(
            "Basin memory",
            r.get("peak_K_basin_memory", np.nan),
            "basin memory score",
            r.get("peak_basin_memory_score", np.nan),
            "memory",
            source
        )

    # Hysteresis
    elif folder == "stuart_landau_hysteresis_output":
        add(
            "Hysteresis",
            r.get("K_at_max_cluster_gap", np.nan),
            "max cluster hysteresis gap",
            r.get("max_cluster_gap", np.nan),
            "path dependence",
            source
        )

    # Recovery
    elif folder == "stuart_landau_recovery_output":
        add(
            "Recovery delay",
            r.get("peak_K_recovery_time", np.nan),
            "peak recovery time",
            r.get("peak_recovery_time", np.nan),
            "recovery",
            source
        )

    # Memory lock
    elif folder == "memory_lock_edge_v2":
        add(
            "Memory-lock edge",
            r.get("peak_K_memory_edge", np.nan),
            "memory edge score",
            r.get("peak_memory_edge_score", np.nan),
            "memory lock",
            source
        )

    # Metastability lifetime
    elif folder == "stuart_landau_metastability_lifetime":
        add(
            "Metastability lifetime",
            r.get("peak_K_memory_supported_lifetime", np.nan),
            "memory-supported lifetime",
            r.get("peak_memory_supported_lifetime", np.nan),
            "metastability",
            source
        )

    # Adaptive coupling
    elif folder == "stuart_landau_adaptive_coupling_v1":
        add(
            "Adaptive coupling",
            r.get("peak_K_adaptive_reorganization", np.nan),
            "adaptive reorganization score",
            r.get("peak_adaptive_reorganization_score", np.nan),
            "adaptive self-organization",
            source
        )

    # Network topology
    elif folder == "stuart_landau_network_topology_v1":
        # one row per topology if summary has many rows
        full = pd.read_csv(r["path"])
        for _, rr in full.iterrows():
            add(
                "Topology: " + str(rr.get("topology")),
                rr.get("peak_K_reorganization", np.nan),
                "topology reorganization score",
                rr.get("peak_reorganization_score", np.nan),
                "topology robustness",
                source
            )

    # Irreversibility corrected
    elif folder == "stuart_landau_irreversibility_threshold_v1" and "global_corrected" in source:
        add(
            "Irreversibility global-corrected",
            r.get("peak_K_excess_global_irreversibility", np.nan),
            "excess global irreversibility",
            r.get("peak_excess_global_state_irreversibility", np.nan),
            "irreversibility",
            source
        )

    # Outsider rebellion
    elif folder == "stuart_landau_outsider_rebellion_output":
        full = pd.read_csv(r["path"])
        if "peak_takeover_fraction" in full.columns:
            max_row = full.sort_values("peak_takeover_fraction", ascending=False).iloc[0]
            add(
                "Outsider rebellion",
                max_row.get("K_at_peak_takeover", np.nan),
                "peak takeover fraction",
                max_row.get("peak_takeover_fraction", np.nan),
                "outsider dynamics",
                source
            )

    # Outsider recruitment
    elif folder == "stuart_landau_outsider_recruitment_v1":
        add(
            "Outsider recruitment",
            r.get("peak_K_outsider_recruitment", np.nan),
            "outsider recruitment score",
            r.get("peak_outsider_recruitment_score", np.nan),
            "outsider assimilation",
            source
        )

    # Null/control
    elif folder == "stuart_landau_null_control_v1":
        full = pd.read_csv(r["path"])
        for _, rr in full.iterrows():
            add(
                "Null/control: " + str(rr.get("control_type")),
                rr.get("peak_K_reorganization", np.nan),
                "control reorganization score",
                rr.get("peak_reorganization_score", np.nan),
                "control",
                source
            )

    # Metastability landscape
    elif folder == "stuart_landau_metastability_landscape_v1":
        add(
            "Metastability landscape control",
            np.nan,
            "mean hysteresis gap",
            r.get("mean_hysteresis_gap", np.nan),
            "weak/negative control",
            source
        )

unified = pd.DataFrame(rows)

unified.to_csv(
    os.path.join(OUT, "unified_phase_diagram_summary.csv"),
    index=False
)

# ------------------------------------------------------------
# Window statistics
# ------------------------------------------------------------

valid = unified.dropna(subset=["K_peak_or_threshold"]).copy()

window_stats = pd.DataFrame([{
    "K_min": valid["K_peak_or_threshold"].min(),
    "K_max": valid["K_peak_or_threshold"].max(),
    "K_mean": valid["K_peak_or_threshold"].mean(),
    "K_median": valid["K_peak_or_threshold"].median(),
    "K_std": valid["K_peak_or_threshold"].std(),
    "n_tests_with_K": len(valid)
}])

window_stats.to_csv(
    os.path.join(OUT, "unified_window_statistics.csv"),
    index=False
)

# ------------------------------------------------------------
# Figure 1: all peak K values
# ------------------------------------------------------------

plot_df = valid.sort_values("K_peak_or_threshold").reset_index(drop=True)

plt.figure(figsize=(11, max(5, 0.35 * len(plot_df))))

y = np.arange(len(plot_df))

plt.scatter(plot_df["K_peak_or_threshold"], y, s=90)

plt.yticks(y, plot_df["test"])
plt.xlabel("K peak / threshold")
plt.title("Unified Stuart–Landau phase diagram: peak positions")

plt.axvspan(0.45, 0.70, alpha=0.12, label="core reorganization window 0.45–0.70")
plt.axvspan(0.70, 0.85, alpha=0.08, label="lock / adaptive transition zone")
plt.axvline(plot_df["K_peak_or_threshold"].median(), linestyle="--", label=f"median K={plot_df['K_peak_or_threshold'].median():.3f}")

plt.legend()
plt.tight_layout()

plt.savefig(
    os.path.join(OUT, "fig_unified_peak_K_positions.png"),
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# Figure 2: category-level phase map
# ------------------------------------------------------------

cat_df = (
    valid.groupby("category")
    .agg({
        "K_peak_or_threshold": ["mean", "median", "min", "max", "count"]
    })
)

cat_df.columns = ["_".join(c) for c in cat_df.columns]
cat_df = cat_df.reset_index()

cat_df.to_csv(
    os.path.join(OUT, "unified_category_phase_summary.csv"),
    index=False
)

plt.figure(figsize=(10, 5))

for _, row in cat_df.iterrows():
    plt.plot(
        [row["K_peak_or_threshold_min"], row["K_peak_or_threshold_max"]],
        [row["category"], row["category"]],
        linewidth=4
    )
    plt.scatter(row["K_peak_or_threshold_mean"], row["category"], s=120)

plt.xlabel("K")
plt.ylabel("Category")
plt.title("Unified Stuart–Landau phase map by mechanism category")
plt.tight_layout()

plt.savefig(
    os.path.join(OUT, "fig_unified_category_phase_map.png"),
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# Figure 3: histogram of peak K
# ------------------------------------------------------------

plt.figure(figsize=(8,5))

plt.hist(valid["K_peak_or_threshold"], bins=np.arange(0.35, 1.31, 0.05))

plt.xlabel("K peak / threshold")
plt.ylabel("Count")
plt.title("Distribution of peak K values across Stuart–Landau tests")

plt.tight_layout()

plt.savefig(
    os.path.join(OUT, "fig_unified_peak_K_histogram.png"),
    dpi=300
)

plt.show()

# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("Saved to:")
print(OUT)

print("\nUNIFIED SUMMARY")
display(unified)

print("\nWINDOW STATS")
display(window_stats)

print("\nCATEGORY SUMMARY")
display(cat_df)

In [ ]:
# ============================================================
# STUART-LANDAU MINIMAL REPLICATION v1 — CHECKPOINT
# Purpose: replicate core reorganization window under small parameter changes
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import os

ROOT = "notebooks/Kuramoto"
OUT = ROOT + "/stuart_landau_minimal_replication_v1_CHECKPOINT"
os.makedirs(OUT, exist_ok=True)

CHECKPOINT = OUT + "/minimal_replication_all_checkpoint.csv"

print("Saving to:", OUT)

# ============================================================
# PARAMETERS
# ============================================================

REPLICATIONS = [
    {"replication": "baseline_shifted_seeds", "N": 400, "dt": 0.03,  "omega_std": 0.4,  "seed_offset": 1000},
    {"replication": "lower_noise",           "N": 400, "dt": 0.03,  "omega_std": 0.35, "seed_offset": 2000},
    {"replication": "higher_noise",          "N": 400, "dt": 0.03,  "omega_std": 0.45, "seed_offset": 3000},
    {"replication": "smaller_dt",            "N": 400, "dt": 0.025, "omega_std": 0.4,  "seed_offset": 4000},
    {"replication": "smaller_N",             "N": 300, "dt": 0.03,  "omega_std": 0.4,  "seed_offset": 5000},
]

SEEDS = list(range(10))
K_VALUES = np.round(np.arange(0.4, 0.901, 0.025), 3)

steps = 3000
discard = 1000

alpha = 1.0
cluster_threshold = 0.15

# ============================================================
# HELPERS
# ============================================================

def order_parameter(theta):
    return float(np.abs(np.mean(np.exp(1j * theta))))

def count_clusters(theta, threshold=0.15):
    th = np.sort(np.mod(theta, 2*np.pi))
    gaps = np.diff(th)
    circular_gap = (th[0] + 2*np.pi) - th[-1]
    gaps = np.append(gaps, circular_gap)
    return int(np.sum(gaps > threshold))

def memory_score(cluster_series, window=200):
    s = pd.Series(cluster_series)
    return float((s.rolling(window).std().fillna(0) < 0.2).mean())

# ============================================================
# LOAD CHECKPOINT
# ============================================================

if os.path.exists(CHECKPOINT):
    all_df = pd.read_csv(CHECKPOINT)
    rows = all_df.to_dict("records")
    done = set(zip(
        all_df["replication"],
        all_df["K"].round(3),
        all_df["seed"]
    ))
    print("Loaded checkpoint rows:", len(rows))
else:
    rows = []
    done = set()
    print("No checkpoint found. Starting fresh.")

# ============================================================
# MAIN LOOP
# ============================================================

for cfg in REPLICATIONS:

    rep_name = cfg["replication"]
    N = cfg["N"]
    dt = cfg["dt"]
    omega_std = cfg["omega_std"]
    seed_offset = cfg["seed_offset"]

    print("\n" + "="*70)
    print("REPLICATION:", rep_name, "| N:", N, "| dt:", dt, "| omega_std:", omega_std)
    print("="*70)

    for K in K_VALUES:

        print(f"\nK={K}")

        for seed in tqdm(SEEDS):

            actual_seed = seed + seed_offset
            key = (rep_name, round(float(K), 3), int(seed))

            if key in done:
                continue

            rng = np.random.default_rng(actual_seed)

            omega = rng.normal(0, omega_std, N)

            z = (
                rng.normal(0, 0.5, N)
                + 1j * rng.normal(0, 0.5, N)
            )

            R_series = []
            cluster_series = []

            for t in range(steps):

                mean_z = np.mean(z)

                dz = (
                    (alpha + 1j * omega - np.abs(z)**2) * z
                    + K * (mean_z - z)
                )

                z = z + dt * dz

                if t >= discard:
                    theta = np.angle(z)
                    R_series.append(order_parameter(theta))
                    cluster_series.append(count_clusters(theta, cluster_threshold))

            R_series = np.array(R_series)
            cluster_series = np.array(cluster_series)

            dominant_cluster = pd.Series(cluster_series).mode().iloc[0]
            dominant_fraction = float(np.mean(cluster_series == dominant_cluster))

            switching_rate = float(np.sum(np.diff(cluster_series) != 0) / len(cluster_series))

            reorganization_score = float(
                np.std(cluster_series)
                * (1 - dominant_fraction)
                * (switching_rate + 1e-9)
            )

            row = {
                "replication": rep_name,
                "N": N,
                "dt": dt,
                "omega_std": omega_std,
                "K": K,
                "seed": seed,
                "actual_seed": actual_seed,

                "R_mean": float(np.mean(R_series)),
                "R_std": float(np.std(R_series)),

                "cluster_mean": float(np.mean(cluster_series)),
                "cluster_std": float(np.std(cluster_series)),

                "dominant_cluster": int(dominant_cluster),
                "dominant_fraction": dominant_fraction,
                "switching_rate": switching_rate,
                "memory_score": memory_score(cluster_series),

                "reorganization_score": reorganization_score
            }

            rows.append(row)
            done.add(key)

            pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)

print("\nFULL LOOP DONE")

# ============================================================
# AGGREGATE
# ============================================================

all_df = pd.DataFrame(rows)

agg = (
    all_df
    .groupby(["replication", "K"])
    .agg({
        "R_mean": ["mean", "std"],
        "R_std": ["mean", "std"],
        "cluster_mean": ["mean", "std"],
        "cluster_std": ["mean", "std"],
        "dominant_fraction": ["mean", "std"],
        "switching_rate": ["mean", "std"],
        "memory_score": ["mean", "std"],
        "reorganization_score": ["mean", "std"],
    })
)

agg.columns = ["_".join(c) for c in agg.columns]
agg = agg.reset_index()

summary_rows = []

for rep_name in agg["replication"].unique():

    sub = agg[agg["replication"] == rep_name].copy()
    idx = sub["reorganization_score_mean"].idxmax()

    summary_rows.append({
        "replication": rep_name,
        "peak_K_reorganization": sub.loc[idx, "K"],
        "peak_reorganization_score": sub.loc[idx, "reorganization_score_mean"],
        "R_mean_at_peak": sub.loc[idx, "R_mean_mean"],
        "cluster_mean_at_peak": sub.loc[idx, "cluster_mean_mean"],
        "dominant_fraction_at_peak": sub.loc[idx, "dominant_fraction_mean"],
        "switching_rate_at_peak": sub.loc[idx, "switching_rate_mean"],
        "memory_score_at_peak": sub.loc[idx, "memory_score_mean"],
    })

summary = pd.DataFrame(summary_rows)

replication_window = pd.DataFrame([{
    "K_min_peak": summary["peak_K_reorganization"].min(),
    "K_max_peak": summary["peak_K_reorganization"].max(),
    "K_mean_peak": summary["peak_K_reorganization"].mean(),
    "K_median_peak": summary["peak_K_reorganization"].median(),
    "K_std_peak": summary["peak_K_reorganization"].std(),
    "n_replications": len(summary)
}])

# ============================================================
# SAVE
# ============================================================

all_df.to_csv(OUT + "/minimal_replication_all.csv", index=False)
agg.to_csv(OUT + "/minimal_replication_aggregate.csv", index=False)
summary.to_csv(OUT + "/minimal_replication_summary.csv", index=False)
replication_window.to_csv(OUT + "/minimal_replication_window.csv", index=False)

# ============================================================
# FIGURES
# ============================================================

plt.figure(figsize=(10,6))
for rep_name in agg["replication"].unique():
    sub = agg[agg["replication"] == rep_name]
    plt.plot(sub["K"], sub["reorganization_score_mean"], marker="o", label=rep_name)

plt.axvspan(0.55, 0.70, alpha=0.12, label="expected core window 0.55–0.70")
plt.xlabel("K")
plt.ylabel("Reorganization score")
plt.title("Minimal replication: reorganization score")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_minimal_replication_reorganization_score.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for rep_name in agg["replication"].unique():
    sub = agg[agg["replication"] == rep_name]
    plt.plot(sub["K"], sub["switching_rate_mean"], marker="o", label=rep_name)

plt.xlabel("K")
plt.ylabel("Switching rate")
plt.title("Minimal replication: switching rate")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_minimal_replication_switching_rate.png", dpi=300)
plt.show()

plt.figure(figsize=(10,6))
for rep_name in agg["replication"].unique():
    sub = agg[agg["replication"] == rep_name]
    plt.plot(sub["K"], sub["R_mean_mean"], marker="o", label=rep_name)

plt.xlabel("K")
plt.ylabel("R mean")
plt.title("Minimal replication: synchronization curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_minimal_replication_R_vs_K.png", dpi=300)
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(summary["peak_K_reorganization"], summary["replication"], s=120)
plt.axvspan(0.55, 0.70, alpha=0.12)
plt.xlabel("Peak K")
plt.ylabel("Replication")
plt.title("Minimal replication: peak K stability")
plt.grid(True)
plt.tight_layout()
plt.savefig(OUT + "/fig_minimal_replication_peak_K_summary.png", dpi=300)
plt.show()

# ============================================================
# DISPLAY
# ============================================================

print("\nSUMMARY")
display(summary)

print("\nREPLICATION WINDOW")
display(replication_window)

print("\nAGGREGATE PREVIEW")
display(agg.head(30))

print("\nSaved to:")
print(OUT)

In [ ]:
import os
import shutil
import pandas as pd
from pathlib import Path

MASTER_DIR = "notebooks/Kuramoto/MASTER_STUART_LANDAU_ARCHIVE"

SOURCE = "notebooks/Kuramoto/stuart_landau_minimal_replication_v1_CHECKPOINT"
TARGET = os.path.join(MASTER_DIR, "stuart_landau_minimal_replication_v1")

os.makedirs(TARGET, exist_ok=True)

for f in os.listdir(SOURCE):
    src = os.path.join(SOURCE, f)
    dst = os.path.join(TARGET, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)

rows = []

for path, dirs, files in os.walk(MASTER_DIR):
    for f in files:
        if f.endswith((".csv", ".png", ".docx", ".pdf")):
            full = os.path.join(path, f)
            rel = os.path.relpath(full, MASTER_DIR)
            parts = Path(rel).parts

            rows.append({
                "source": parts[0] if len(parts) > 0 else "",
                "folder": parts[-2] if len(parts) > 1 else "",
                "filename": f,
                "extension": Path(f).suffix.lower(),
                "size_kb": round(os.path.getsize(full) / 1024, 2),
                "path": full
            })

index_df = pd.DataFrame(rows).sort_values(
    ["folder", "extension", "filename"]
).reset_index(drop=True)

index_df.to_csv(os.path.join(MASTER_DIR, "MASTER_INDEX.csv"), index=False)

summary = (
    index_df.groupby(["folder", "extension"])
    .size()
    .reset_index(name="file_count")
    .sort_values(["folder", "extension"])
)

summary.to_csv(os.path.join(MASTER_DIR, "MASTER_INDEX_SUMMARY.csv"), index=False)

print("DONE")
print("MASTER_INDEX updated")
print("Total files:", len(index_df))
display(summary)